In [1]:
root_path = "/gpfs/home/pl2948/evo2/"

https://www.nature.com/articles/s41586-022-04558-8#data-availability

# ClinVar tab txt file

In [2]:
import os
import pandas as pd

ClinVar = pd.read_csv(os.path.join(root_path, "ClinVar", "variant_summary_2026-02.txt.gz"), 
                      sep='\t', dtype={'Chromosome': str}, low_memory=False, compression='gzip') 

display(ClinVar)

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8674140,4799520,Duplication,NM_001458.5(FLNC):c.3398dup (p.Leu1134fs),2318,FLNC,HGNC:3756,Likely pathogenic,1,"Nov 21, 2025",-1,...,AT,-,-,-,-,-,-,SCV007338517,-,-
8674141,4799521,single nucleotide variant,NM_001035.3(RYR2):c.4411G>A (p.Asp1471Asn),6262,RYR2,HGNC:10484,Uncertain significance,0,"Nov 21, 2025",-1,...,A,-,-,-,-,-,-,SCV007338518,-,-
8674142,4799521,single nucleotide variant,NM_001035.3(RYR2):c.4411G>A (p.Asp1471Asn),6262,RYR2,HGNC:10484,Uncertain significance,0,"Nov 21, 2025",-1,...,A,-,-,-,-,-,-,SCV007338518,-,-
8674143,4799522,single nucleotide variant,NM_015560.3:c.887A>G,4976,OPA1,HGNC:8140,Pathogenic,1,"Jan 29, 2026",-1,...,na,-,-,-,-,-,-,SCV007338530,-,-


In [3]:
for col in ClinVar.columns:
    print(col)
    print(ClinVar[col].unique())
    print()

#AlleleID
[  15041   15042   15043 ... 4799521 4799522 4799523]

Type
<ArrowStringArray>
[                    'Indel',                  'Deletion',
 'single nucleotide variant',               'Duplication',
            'Microsatellite',                 'Insertion',
                 'Variation',                   'Complex',
             'Translocation',                 'Inversion',
          'copy number gain',                    'fusion',
          'copy number loss',              'protein only',
        'Tandem duplication']
Length: 15, dtype: str

Name
<ArrowStringArray>
['NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer)',
                                        'NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs)',
                                          'NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg)',
                                           'NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter)',
                                          'NM_017547.4(FOXRE

In [4]:
ClinVar = ClinVar[ClinVar['Assembly'] == 'GRCh38']
print('GRCh38', ClinVar.shape[0])

GRCh38 4303534


In [5]:
ClinVar = ClinVar[ClinVar['OriginSimple']!='somatic'].reset_index(drop=True)
print('Exclude somatic', len(ClinVar), 'remain')

Exclude somatic 4296847 remain


In [6]:
basic_chromosome = list(map(str, range(1,23))) + ['X', 'Y']
ClinVar = ClinVar[ClinVar['Chromosome'].isin(basic_chromosome)]
    
print('Basic Chromosome', ClinVar.shape[0])

Basic Chromosome 4293730


In [7]:
for mut in ClinVar['Type'].unique():
    print(mut, ClinVar[ClinVar['Type']==mut].shape[0])

Indel 17772
Deletion 156508
single nucleotide variant 3985389
Duplication 67014
Microsatellite 37952
Insertion 13496
Variation 442
Translocation 18
Inversion 1441
copy number gain 6945
copy number loss 6744
Complex 9


In [8]:
ClinVar = ClinVar[ClinVar['Type'] == 'single nucleotide variant']
print('single nucleotide variant', ClinVar.shape[0])

single nucleotide variant 3985389


In [9]:
import numpy as np

REVIEW_STATUS_TO_GOLD_STARS = {
    'criteria provided, single submitter': 1,
    'criteria provided, multiple submitters, no conflicts': 2,
    'criteria provided, conflicting interpretations': 1,
    'no assertion criteria provided': np.nan,
    'reviewed by expert panel': 3,
    'no assertion provided': np.nan,
    'no interpretation for the single variant': np.nan,
    'practice guideline': 4,
    }

CLINICAL_SIGNIFICANCE_TO_LABEL = {
    'Benign': 0,
    'Likely benign': 0.1,
    'Benign/Likely benign': 0.1,
    'Pathogenic': 1,
    'Likely pathogenic': 1.1,
    'Pathogenic/Likely pathogenic': 1.1,
    'Uncertain significance': 2
    }

ClinVar['ClinVar_annotation'] = ClinVar['ClinicalSignificance'].map(CLINICAL_SIGNIFICANCE_TO_LABEL)
ClinVar = ClinVar[ClinVar['ClinVar_annotation'].isin({0, 0.1, 1, 1.1})]

ClinVar['gold_stars'] = ClinVar['ReviewStatus'].map(REVIEW_STATUS_TO_GOLD_STARS)
print('Total Number:\t', len(ClinVar))
print('Benign:\t\t', len(ClinVar[ClinVar['ClinVar_annotation']==0]))
print('Likely Benign:\t\t', len(ClinVar[ClinVar['ClinVar_annotation']==0.1]))
print('Pathogenic:\t', len(ClinVar[ClinVar['ClinVar_annotation']==1]))
print('Likely Pathogenic:\t', len(ClinVar[ClinVar['ClinVar_annotation']==1.1]))
# print('VUS:\t\t', len(ClinVar[ClinVar['ClinVar_annotation']==2]))

ClinVar = ClinVar[ClinVar['gold_stars'].isin({1, 2, 3, 4})]
print('Gold Stars >=1 :', len(ClinVar))

Total Number:	 1398129
Benign:		 178910
Likely Benign:		 1041214
Pathogenic:	 82148
Likely Pathogenic:	 95857
Gold Stars >=1 : 1346049


# MANE

## Build annotation reference from MANE

In [ ]:
import os
import gzip
import pandas as pd

with gzip.open(os.path.join(root_path, "MANE", "MANE.GRCh38.v1.5.refseq_genomic.gff.gz"), "rt") as file:
    lines = file.readlines()

columns = ["Chromosome", "Source", "Feature", "Start", "End", "Score", "Strand", "Frame", "Attributes"]
data = []

for line in lines:
    if not line.startswith("#"):
        fields = line.strip().split("\t")
        data.append(fields)

MANE = pd.DataFrame(data, columns=columns)
display(MANE)

In [ ]:
import pandas as pd
import re

fields = [
    'ID', 'Parent', 'Dbxref', 'Name', 'description', 'gbkey', 'gene', 'gene_biotype', 'product', 'tag', 'transcript_id']

def parse_attributes(attribute_string):
    attributes = {}
    for field in fields:
        match = re.search(f'{field}=([^;]+)', attribute_string)
        if match:
            attributes[field] = match.group(1)
    if 'Dbxref' in attributes:
        dbxref_values = attributes['Dbxref'].split(',')
        for value in dbxref_values:
            if value.startswith('Ensembl:'):
                attributes['Ensembl'] = value.split(':')[1]
                break
    return attributes

parsed_attributes = MANE['Attributes'].apply(parse_attributes)
parsed_df = pd.DataFrame(parsed_attributes.tolist())

MANE = pd.concat([MANE, parsed_df], axis=1)

display(MANE)

In [ ]:
MANE.isna().sum()

In [ ]:
print(MANE['Feature'].unique())

for feature in MANE['Feature'].unique():
    parent = MANE[MANE['Feature']==feature]['Parent']
    parent_type = MANE[MANE['ID'].isin(parent)]['Feature'].unique()
    print(f"{feature:<15} number: {sum(MANE['Feature'] == feature):<10} parent: {str(parent_type):<30}")

In [ ]:
import pandas as pd

distance_dict = {'mRNA': 2000, 
                 'lncRNA': 2000,
                 'snoRNA': 500,
                 'snRNA': 500,
                 'telomerase_RNA': 1500,
                 'antisense_RNA': 1500
}
Promoter = MANE[MANE['Feature'].isin(distance_dict.keys())].copy()
Promoter[["Promoter_Start", "Promoter_End"]]=None

def calculate_promoter(row):
    if row["Strand"] == "+":
        tss = int(row["Start"])
        promoter_start = tss - distance_dict[row['Feature']]
        promoter_end = tss
    else: 
        tss = int(row["End"])
        promoter_start = tss
        promoter_end = tss + distance_dict[row['Feature']]
    return promoter_start, promoter_end

Promoter[["Promoter_Start", "Promoter_End"]] = Promoter.apply(calculate_promoter, axis=1, result_type="expand")
Promoter = Promoter.reset_index(drop=True)

display(Promoter)

# Annotate with MANE

In [ ]:
columns_to_keep = ['Name', 'GeneSymbol', 'RS# (dbSNP)', 'RCVaccession', 
                   'PhenotypeIDS', 'PhenotypeList', 'Assembly', 
                   'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
                   'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'ClinVar_annotation', 'gold_stars']

ClinVar_annotation = ClinVar[columns_to_keep].copy().reset_index(drop=True)
ClinVar_annotation

gene:
- mRNA: CDS, exon
- lnc_RNA: exon
- snRNA: exon
- antisense_RNA: exon
- telomerase_RNA: exon
- RNase_MRP_RNA: exon
- snoRNA: exon  
['gene' 'mRNA' 'exon' 'CDS' 'lnc_RNA' 'snRNA' 'antisense_RNA'
 'telomerase_RNA' 'RNase_MRP_RNA' 'snoRNA']

In [ ]:
import pandas as pd
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

annotation_columns = ['gene', 
                      'mRNA', 'mRNA_promoter', 'mRNA_exon', 
                            'coding_sequence', 'start_codon', 'stop_codon', 'five_prime_UTR', 'three_prime_UTR', 'mRNA_intron', 'mRNA_splice',
                      'lncRNA', 'lncRNA_promoter', 'lncRNA_exon', 
                      'snRNA', 'snRNA_promoter', 'snRNA_exon', 
                      'antisenseRNA', 'antisenseRNA_promoter', 'antisenseRNA_exon', 
                      'telomeraseRNA', 'telomeraseRNA_promoter', 'telomeraseRNA_exon', 
                      'RNaseMRPRNA', 'RNaseMRPRNA_promoter', 'RNaseMRPRNA_exon', 
                      'snoRNA', 'snoRNA_promoter', 'snoRNA_exon', 
                      'other']

def match_variants(row):
    chrom = row["Chromosome"]
    pos = int(row["PositionVCF"])
    annotation = MANE[((MANE["Chromosome"] == f'chr{chrom}') & 
                       (MANE["Start"].astype(int) <= pos) & 
                       (MANE["End"].astype(int) >= pos))]
    
    annotation_promoter = Promoter[((Promoter["Chromosome"] == chrom) & 
                                    (Promoter["Promoter_Start"].astype(int) <= pos) & 
                                    (Promoter["Promoter_End"].astype(int) >= pos))]
    
    if annotation.empty and annotation_promoter.empty:
        row['other'] = 1
        return row

    types = annotation['Feature'].unique()
    types_promoter = annotation_promoter['Feature'].unique()
    
    if "gene" in types:
        row['gene'] = 1
    
    if "mRNA" in types:
        row['mRNA'] = 1
        transcript_ids = set(annotation[annotation['Feature'] == "mRNA"]["transcript_id"].dropna())
        row['transcript_set'].update(transcript_ids)

        for transcript_id in transcript_ids:
            strand = annotation[annotation['ID']==f'rna-{transcript_id}']['Strand'].iloc[0]
            annotation_mRNA_exon = annotation[((annotation['Parent']==f'rna-{transcript_id}') & 
                                               (annotation['Feature'] == "exon"))]
            annotation_mRNA_CDS = annotation[((annotation['Parent']==f'rna-{transcript_id}') & 
                                               (annotation['Feature'] == "CDS"))]
            transcript_exon = MANE[((MANE['Parent']==f'rna-{transcript_id}') & 
                                    (MANE['Feature'] == "exon"))]
            transcript_CDS = MANE[((MANE['Parent']==f'rna-{transcript_id}') & 
                                   (MANE['Feature'] == "CDS"))]

            if (not annotation_mRNA_CDS.empty) and (not annotation_mRNA_exon.empty): # So it is exon and coding sequence
                row['mRNA_exon'] = 1
                row['coding_sequence'] = 1
                
                if strand=="+":
                    start_1 = min(transcript_CDS['Start'].astype(int))
                    if (pos <= start_1+2) and (pos >= start_1):
                        row['start_codon'] = 1
                    stop_3 = max(transcript_CDS['End'].astype(int))
                    if (pos >= stop_3-2) and (pos <= stop_3):
                        row['stop_codon'] = 1
                else: #strand=="-"
                    start_1 = max(transcript_CDS['End'].astype(int))
                    if (pos >= start_1-2) and (pos <= start_1):
                        row['start_codon'] = 1
                    stop_3 = min(transcript_CDS['Start'].astype(int))
                    if (pos <= stop_3+2) and (pos >= stop_3):
                        row['stop_codon'] = 1
                
            elif (annotation_mRNA_CDS.empty) and (not annotation_mRNA_exon.empty): # So it is UTR
                row['mRNA_exon'] = 1
                if strand=="+":
                    fiveUTR_start = min(transcript_exon['Start'].astype(int))
                    fiveUTR_end = min(transcript_CDS['Start'].astype(int))-1
                    if (pos <= fiveUTR_end) and (pos >= fiveUTR_start):
                        row['five_prime_UTR']=1
                    threeUTR_start = max(transcript_CDS['End'].astype(int))+1
                    threeUTR_end = max(transcript_exon['End'].astype(int))
                    if (pos >= threeUTR_start) and (pos <= threeUTR_end):
                        row['three_prime_UTR']=1
                else: #strand=="-"
                    fiveUTR_start = max(transcript_exon['End'].astype(int))
                    fiveUTR_end = max(transcript_CDS['End'].astype(int))+1
                    if (pos >= fiveUTR_end) and (pos <= fiveUTR_start):
                        row['five_prime_UTR']=1
                    threeUTR_start = min(transcript_CDS['Start'].astype(int))-1
                    threeUTR_end = min(transcript_exon['Start'].astype(int))
                    if (pos <= threeUTR_start) and (pos >= threeUTR_end):
                        row['three_prime_UTR']=1

            elif (annotation_mRNA_CDS.empty) and (annotation_mRNA_exon.empty): # So it is intron
                row['mRNA_intron'] = 1
                splice_region = transcript_exon.apply(lambda exon: pos in [int(exon['Start'])-1, 
                                                                           int(exon['Start'])-2, 
                                                                           int(exon['End'])+1, 
                                                                           int(exon['End'])+2], axis=1)
                if splice_region.any():
                    row['mRNA_splice'] = 1

    if "mRNA" in types_promoter:
        row['mRNA_promoter'] = 1
        transcript_ids = set(annotation_promoter[annotation_promoter['Feature'] == "mRNA"]["transcript_id"].dropna())
        row['promoter_transcript_set'].update(transcript_ids)
        
    for rna in ['lncRNA', 'snRNA', 'antisenseRNA', 'telomeraseRNA', 'RNaseMRPRNA', 'snoRNA']:
        if rna in types:
            row[rna]=1
            transcript_ids = set(annotation[annotation['Feature'] == rna]["transcript_id"].dropna())
            row['transcript_set'].update(transcript_ids)
            
            for transcript_id in transcript_ids:
                annotation_RNA_exon = annotation[((annotation['Parent']==f'rna-{transcript_id}') & 
                                                   (annotation['Feature'] == "exon"))]
                if not annotation_RNA_exon.empty:
                    row[f"{rna}_exon"] = 1

        if rna in types_promoter:
            row[f"{rna}_promoter"] = 1
            transcript_ids = set(annotation_promoter[annotation_promoter['Feature'] == rna]["transcript_id"].dropna())
            row['promoter_transcript_set'].update(transcript_ids)
    
    return row

In [ ]:
ClinVar_annotation[annotation_columns] = 0
ClinVar_annotation['transcript_set'] = ClinVar_annotation.apply(lambda x: set(), axis=1)
ClinVar_annotation['promoter_transcript_set'] = ClinVar_annotation.apply(lambda x: set(), axis=1)

num_cpus = cpu_count()
with Pool(cpu_count()) as pool:
    results = list(tqdm(pool.imap(match_variants, [row for _, row in ClinVar_annotation.iterrows()]), total=len(ClinVar_annotation)))

ClinVar_annotation = pd.DataFrame(results)

display(ClinVar_annotation)
print(ClinVar_annotation[annotation_columns].sum())

In [ ]:
columns_to_sum = annotation_columns.copy()

column_sums = ClinVar_annotation[columns_to_sum].sum()

print(column_sums)

In [ ]:
import pandas as pd

columns_to_sum = annotation_columns.copy()

ClinVar_annotation['row_sum'] = ClinVar_annotation[columns_to_sum].sum(axis=1)

row_sum_counts = ClinVar_annotation['row_sum'].value_counts().sort_index()

print(row_sum_counts)

In [ ]:
import pandas as pd
import gzip
import os

ClinVar_annotation.to_csv(os.path.join(root_path, 'data', 'AllClinVarBenchmark_202602.csv.gz'), index=False, compression='gzip')

In [ ]:
non_zero_columns = column_sums[column_sums != 0]
print(non_zero_columns)

In [ ]:
import pandas as pd

columns_to_sum = non_zero_columns.index

heatmap_data = pd.DataFrame(index=columns_to_sum, columns=columns_to_sum)

for row in columns_to_sum:
    for column in columns_to_sum:
        count = ClinVar_annotation[(ClinVar_annotation[row] == 1) & (ClinVar_annotation[column] == 1)].shape[0]
        heatmap_data.loc[row, column] = count

# print(heatmap_data)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = heatmap_data.astype(int)

plt.figure(figsize=(18, 18))
sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu")
plt.title('Heatmap of Feature Combinations')
plt.show()

In [ ]:
max_values = heatmap_data.max()

percentage_data = heatmap_data.div(max_values, axis=1) * 100

plt.figure(figsize=(18, 18))
sns.heatmap(percentage_data, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title('Percentage Heatmap of Feature Combinations (%)')
plt.show()

# Add other information

## Get splicing

In [ ]:
import pandas as pd

pattern = r"\d+(\+1|\+2|\-1|\-2)[ATCG]"

ClinVar_annotation['ClinVarName_splice'] = 0

ClinVar_annotation['ClinVarName_splice'] = ClinVar_annotation['Name'].str.contains(pattern, na=False).astype(int)

ClinVar_annotation['ClinVarName_splice'].sum()

## Get noncoding RNA

In [ ]:
import pandas as pd

pattern = r"NR_"

ClinVar_annotation['ClinVarName_RNA_gene'] =ClinVar_annotation['Name'].str.contains(pattern, na=False).astype(int)

ClinVar_annotation['ClinVarName_RNA_gene'].sum()

## Get protein information

In [ ]:
from Bio.SeqUtils import seq1
import re

In [ ]:
def ParseAminoAcidChange(variant_name):
    '''
    depend on numpy and re
        import numpy as np
        import re
    Extract from HGVS nomenclature to amino acid change
    '''
    PROTEIN_NAME_PATTERN = re.compile(r'\(p\.(.+)\)')
    protein_changes = PROTEIN_NAME_PATTERN.findall(variant_name)
    
    if len(protein_changes) == 0:
        return np.nan
    else:
        protein_change, = protein_changes
        return protein_change

In [ ]:
def ParseRefSeqTemplate(variant_name):
    '''
    depend on re
        import re
    Extract from HGVS name to Ref-Seq template
    '''
    REFSEQ_ID_PATTERN = re.compile(r'NM_\d+\.?\d*')
    return REFSEQ_ID_PATTERN.findall(variant_name)

In [ ]:
def strict_seq1(aa3):
    '''
    depend on biopython
        from Bio.SeqUtils import seq1
    '''
    aa1 = seq1(aa3)
    VALID_AA1_SET = set('ACDEFGHIKLMNPQRSTVWY*')
    assert aa1 in VALID_AA1_SET, 'Invalid 3-letter (%s) to 1-letter (%s) conversion.' % (aa3, aa1)
    return aa1

def parse_aa_and_pos(aa_and_pos):
    '''
    depend on biopython
        from Bio.SeqUtils import seq1
    '''
    aa = strict_seq1(aa_and_pos[:3])
    pos = int(aa_and_pos[3:])
    return aa, pos

In [ ]:
def series_return(index):
    import pandas as pd
    def _decorator(f):
        def _wrapped(*args, **kwargs):
            return pd.Series(f(*args, **kwargs), index = index)
        return _wrapped
    return _decorator

@series_return(['failed_parsing_protein_change', 'start_pos', 'end_pos', 'start_aa', 'end_aa', 'alt'])
def parse_hgvs_protein_change(raw_protein_change):
    '''
    dependent on numpy, pandas, and re
        import numpy as np
        import pandas as pd
        import re
    Parse HGVS protein change to failed_parsing_protein_change', 'start_pos', 'end_pos', 'start_aa', 'end_aa', 'alt'
    '''
    
    if pd.isnull(raw_protein_change):
        return False, np.nan, np.nan, np.nan, np.nan, np.nan

    try:

        last_digit_pos = list(re.finditer(r'\d', raw_protein_change))[-1].start()
        raw_ref = raw_protein_change[:(last_digit_pos + 1)]
        raw_alt = raw_protein_change[(last_digit_pos + 1):]

        raw_ref_parts = raw_ref.split('_')
        assert 1 <= len(raw_ref_parts) <= 2
        ref_aas, ref_positions = zip(*map(parse_aa_and_pos, raw_ref_parts))
        start, end = ref_positions[0], ref_positions[-1]
        start_aa, end_aa = ref_aas[0], ref_aas[-1]

        if raw_alt.endswith('fs'):
            alt = 'fs'
        elif raw_alt == '=':
            alt = '='
        elif raw_alt == 'dup':
            alt = 'dup'
        else:
            is_insert = ('ins' in raw_alt) and ('del' not in raw_alt)
            raw_alt_aas = raw_alt.replace('del', '').replace('ins', '')
            alt = ''.join(map(strict_seq1, [raw_alt_aas[i:(i + 3)] for i in range(0, len(raw_alt_aas), 3)]))

            if is_insert:
                assert end == start + 1
                alt = start_aa + alt + end_aa

        return False, start, end, start_aa, end_aa, alt
    except:
        return True, np.nan, np.nan, np.nan, np.nan, np.nan

def variant_start_and_end_to_ref_seq(variant):
    if variant['start_pos'] == variant['end_pos']:
        assert variant['start_aa'] == variant['end_aa']
        return variant['start_aa']
    else:
        assert variant['start_pos'] < variant['end_pos'], 'Invalid positions %s-%s' % \
                (variant['start_pos'], variant['end_pos'])
        return variant['start_aa'] + '.*' + variant['end_aa']

In [ ]:
import numpy as np

ClinVar_annotation.loc[:,'ClinVarName_raw_protein_change'] = ClinVar_annotation['Name'].apply(ParseAminoAcidChange)
ClinVar_annotation[['ClinVarName_failed_parsing_protein_change', 
                    'ClinVarName_start_pos', 
                    'ClinVarName_end_pos', 
                    'ClinVarName_start_aa', 
                    'ClinVarName_end_aa', 
                    'ClinVarName_alt']] = ClinVar_annotation['ClinVarName_raw_protein_change'].apply(lambda x: pd.Series(parse_hgvs_protein_change(x)))

ClinVar_annotation['ClinVarName_refseq_ids'] = ClinVar_annotation['Name'].apply(ParseRefSeqTemplate)

In [ ]:
ClinVar_annotation['ClinVarName_refseq_ids'] = ClinVar_annotation['ClinVarName_refseq_ids'].apply(lambda x: x[0] if len(x) > 0 else None)

In [ ]:
ClinVar_annotation['ClinVarName_start_pos'] = pd.to_numeric(ClinVar_annotation['ClinVarName_start_pos'], errors='coerce').astype('Int64')
ClinVar_annotation['ClinVarName_end_pos'] = pd.to_numeric(ClinVar_annotation['ClinVarName_end_pos'], errors='coerce').astype('Int64')

In [ ]:
import pandas as pd

ClinVar_annotation['ClinVarName_coding_sequence'] = (
    ClinVar_annotation[['ClinVarName_start_pos', 'ClinVarName_start_aa', 'ClinVarName_alt']]
    .notna()
    .all(axis=1)  
    .astype(int) 
)

display(ClinVar_annotation[['ClinVarName_start_pos', 'ClinVarName_start_aa', 'ClinVarName_alt', 'ClinVarName_coding_sequence']])

In [ ]:
import pandas as pd
import gzip

ClinVar_annotation.to_csv(os.path.join(root_path, 'data', 'AllClinVarBenchmark_202602.csv.gz'), index=False, compression='gzip')

# Filtering and convert to VCF small benchmark

In [10]:
import pandas as pd
import os

ClinVar_annotation = pd.read_csv(os.path.join(root_path, 'data', 'AllClinVarBenchmark_202602.csv.gz'), 
                                 compression='gzip', low_memory = False)

display(ClinVar_annotation)

,Name,GeneSymbol,RS# (dbSNP),RCVaccession,PhenotypeIDS,PhenotypeList,Assembly,ChromosomeAccession,Chromosome,Start,...,ClinVarName_RNA_gene,ClinVarName_raw_protein_change,ClinVarName_failed_parsing_protein_change,ClinVarName_start_pos,ClinVarName_end_pos,ClinVarName_start_aa,ClinVarName_end_aa,ClinVarName_alt,ClinVarName_refseq_ids,ClinVarName_coding_sequence
0,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),FOXRED1,267606829,RCV000000015|RCV000578659|RCV001194045|RCV0033...,"MONDO:MONDO:0032624,MedGen:C4748791,OMIM:61824...","Mitochondrial complex I deficiency, nuclear ty...",GRCh38,NC_000011.10,11,126275389,...,0,Gln232Ter,False,232.0,232.0,Q,Q,*,NM_017547.4,1
1,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),FOXRED1,267606830,RCV000000016,"MONDO:MONDO:0032624,MedGen:C4748791,OMIM:618241","Mitochondrial complex I deficiency, nuclear ty...",GRCh38,NC_000011.10,11,126277517,...,0,Asn430Ser,False,430.0,430.0,N,N,S,NM_017547.4,1
2,NM_000410.4(HFE):c.892+48G>A,HFE,1800758,RCV000000031|RCV001618204,|MedGen:C3661900,HFE INTRONIC POLYMORPHISM|not provided,GRCh38,NC_000006.12,6,26093008,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_000410.4,0
3,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),HFE,111033563,RCV000000036|RCV001050090|RCV003884332|RCV0052...,"MONDO:MONDO:0021001,MedGen:C3469186,OMIM:23520...",Hemochromatosis type 1|Hereditary hemochromato...,GRCh38,NC_000006.12,6,26092916,...,0,Gln283Pro,False,283.0,283.0,Q,Q,P,NM_000410.4,1
4,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),WDR35,267607175,RCV000000040|RCV000508347,"MONDO:MONDO:0013323,MedGen:C3150874,OMIM:61361...",Cranioectodermal dysplasia 2|not provided,GRCh38,NC_000002.12,2,19933469,...,0,Ala864Thr,False,864.0,864.0,A,A,T,NM_020779.4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346044,NM_001374353.1(GLI2):c.148+1G>A,GLI2,-1,RCV006455030,"MONDO:MONDO:0012563,MedGen:C1835819,OMIM:61082...",Holoprosencephaly 9,GRCh38,NC_000002.12,2,120797469,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_001374353.1,0
1346045,NM_014712.3(SETD1A):c.4582-2A>G,SETD1A,-1,RCV006455031,"MONDO:MONDO:0030005,MedGen:C5882670,OMIM:618832","Epilepsy, early-onset, with or without develop...",GRCh38,NC_000016.10,16,30980737,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_014712.3,0
1346046,NM_005120.3(MED12):c.1540T>G (p.Cys514Gly),MED12,-1,RCV006455032,"MONDO:MONDO:0010590,MedGen:C5399762,OMIM:30545...",FG syndrome 1,GRCh38,NC_000023.11,X,71123149,...,0,Cys514Gly,False,514.0,514.0,C,C,G,NM_005120.3,1
1346047,NM_001374828.1(ARID1B):c.3584G>A (p.Trp1195Ter),ARID1B,-1,RCV006455033,"MONDO:MONDO:0007617,MedGen:C3281201,OMIM:13590...",Coffin-Siris syndrome 1,GRCh38,NC_000006.12,6,157181048,...,0,Trp1195Ter,False,1195.0,1195.0,W,W,*,NM_001374828.1,1


In [11]:
# Filtered_ClinVar = ClinVar_annotation[(ClinVar_annotation['ClinVar_annotation'].isin({0, 1}))].copy().reset_index(drop=True)
Filtered_ClinVar = ClinVar_annotation[(ClinVar_annotation['ClinVar_annotation'].isin({0, 1, 1.1, 0.1}))].copy().reset_index(drop=True)
Filtered_ClinVar

,Name,GeneSymbol,RS# (dbSNP),RCVaccession,PhenotypeIDS,PhenotypeList,Assembly,ChromosomeAccession,Chromosome,Start,...,ClinVarName_RNA_gene,ClinVarName_raw_protein_change,ClinVarName_failed_parsing_protein_change,ClinVarName_start_pos,ClinVarName_end_pos,ClinVarName_start_aa,ClinVarName_end_aa,ClinVarName_alt,ClinVarName_refseq_ids,ClinVarName_coding_sequence
0,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),FOXRED1,267606829,RCV000000015|RCV000578659|RCV001194045|RCV0033...,"MONDO:MONDO:0032624,MedGen:C4748791,OMIM:61824...","Mitochondrial complex I deficiency, nuclear ty...",GRCh38,NC_000011.10,11,126275389,...,0,Gln232Ter,False,232.0,232.0,Q,Q,*,NM_017547.4,1
1,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),FOXRED1,267606830,RCV000000016,"MONDO:MONDO:0032624,MedGen:C4748791,OMIM:618241","Mitochondrial complex I deficiency, nuclear ty...",GRCh38,NC_000011.10,11,126277517,...,0,Asn430Ser,False,430.0,430.0,N,N,S,NM_017547.4,1
2,NM_000410.4(HFE):c.892+48G>A,HFE,1800758,RCV000000031|RCV001618204,|MedGen:C3661900,HFE INTRONIC POLYMORPHISM|not provided,GRCh38,NC_000006.12,6,26093008,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_000410.4,0
3,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),HFE,111033563,RCV000000036|RCV001050090|RCV003884332|RCV0052...,"MONDO:MONDO:0021001,MedGen:C3469186,OMIM:23520...",Hemochromatosis type 1|Hereditary hemochromato...,GRCh38,NC_000006.12,6,26092916,...,0,Gln283Pro,False,283.0,283.0,Q,Q,P,NM_000410.4,1
4,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),WDR35,267607175,RCV000000040|RCV000508347,"MONDO:MONDO:0013323,MedGen:C3150874,OMIM:61361...",Cranioectodermal dysplasia 2|not provided,GRCh38,NC_000002.12,2,19933469,...,0,Ala864Thr,False,864.0,864.0,A,A,T,NM_020779.4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346044,NM_001374353.1(GLI2):c.148+1G>A,GLI2,-1,RCV006455030,"MONDO:MONDO:0012563,MedGen:C1835819,OMIM:61082...",Holoprosencephaly 9,GRCh38,NC_000002.12,2,120797469,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_001374353.1,0
1346045,NM_014712.3(SETD1A):c.4582-2A>G,SETD1A,-1,RCV006455031,"MONDO:MONDO:0030005,MedGen:C5882670,OMIM:618832","Epilepsy, early-onset, with or without develop...",GRCh38,NC_000016.10,16,30980737,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_014712.3,0
1346046,NM_005120.3(MED12):c.1540T>G (p.Cys514Gly),MED12,-1,RCV006455032,"MONDO:MONDO:0010590,MedGen:C5399762,OMIM:30545...",FG syndrome 1,GRCh38,NC_000023.11,X,71123149,...,0,Cys514Gly,False,514.0,514.0,C,C,G,NM_005120.3,1
1346047,NM_001374828.1(ARID1B):c.3584G>A (p.Trp1195Ter),ARID1B,-1,RCV006455033,"MONDO:MONDO:0007617,MedGen:C3281201,OMIM:13590...",Coffin-Siris syndrome 1,GRCh38,NC_000006.12,6,157181048,...,0,Trp1195Ter,False,1195.0,1195.0,W,W,*,NM_001374828.1,1


In [12]:
print('Pathogenic', sum(Filtered_ClinVar['ClinVar_annotation']==1))
print('Likely Pathogenic', sum(Filtered_ClinVar['ClinVar_annotation']==1.1))
print('Benign', sum(Filtered_ClinVar['ClinVar_annotation']==0))
print('Likely Benign', sum(Filtered_ClinVar['ClinVar_annotation']==0.1))

Pathogenic 69519
Likely Pathogenic 89296
Benign 172622
Likely Benign 1014612


In [13]:
Filtered_ClinVar = Filtered_ClinVar[((Filtered_ClinVar['ReferenceAlleleVCF'].isin(['T', 'A', 'C', 'G'])) & 
                                     (Filtered_ClinVar['AlternateAlleleVCF'].isin(['T', 'A', 'C', 'G'])))].reset_index(drop=True)
Filtered_ClinVar

,Name,GeneSymbol,RS# (dbSNP),RCVaccession,PhenotypeIDS,PhenotypeList,Assembly,ChromosomeAccession,Chromosome,Start,...,ClinVarName_RNA_gene,ClinVarName_raw_protein_change,ClinVarName_failed_parsing_protein_change,ClinVarName_start_pos,ClinVarName_end_pos,ClinVarName_start_aa,ClinVarName_end_aa,ClinVarName_alt,ClinVarName_refseq_ids,ClinVarName_coding_sequence
0,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),FOXRED1,267606829,RCV000000015|RCV000578659|RCV001194045|RCV0033...,"MONDO:MONDO:0032624,MedGen:C4748791,OMIM:61824...","Mitochondrial complex I deficiency, nuclear ty...",GRCh38,NC_000011.10,11,126275389,...,0,Gln232Ter,False,232.0,232.0,Q,Q,*,NM_017547.4,1
1,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),FOXRED1,267606830,RCV000000016,"MONDO:MONDO:0032624,MedGen:C4748791,OMIM:618241","Mitochondrial complex I deficiency, nuclear ty...",GRCh38,NC_000011.10,11,126277517,...,0,Asn430Ser,False,430.0,430.0,N,N,S,NM_017547.4,1
2,NM_000410.4(HFE):c.892+48G>A,HFE,1800758,RCV000000031|RCV001618204,|MedGen:C3661900,HFE INTRONIC POLYMORPHISM|not provided,GRCh38,NC_000006.12,6,26093008,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_000410.4,0
3,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),HFE,111033563,RCV000000036|RCV001050090|RCV003884332|RCV0052...,"MONDO:MONDO:0021001,MedGen:C3469186,OMIM:23520...",Hemochromatosis type 1|Hereditary hemochromato...,GRCh38,NC_000006.12,6,26092916,...,0,Gln283Pro,False,283.0,283.0,Q,Q,P,NM_000410.4,1
4,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),WDR35,267607175,RCV000000040|RCV000508347,"MONDO:MONDO:0013323,MedGen:C3150874,OMIM:61361...",Cranioectodermal dysplasia 2|not provided,GRCh38,NC_000002.12,2,19933469,...,0,Ala864Thr,False,864.0,864.0,A,A,T,NM_020779.4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346034,NM_001374353.1(GLI2):c.148+1G>A,GLI2,-1,RCV006455030,"MONDO:MONDO:0012563,MedGen:C1835819,OMIM:61082...",Holoprosencephaly 9,GRCh38,NC_000002.12,2,120797469,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_001374353.1,0
1346035,NM_014712.3(SETD1A):c.4582-2A>G,SETD1A,-1,RCV006455031,"MONDO:MONDO:0030005,MedGen:C5882670,OMIM:618832","Epilepsy, early-onset, with or without develop...",GRCh38,NC_000016.10,16,30980737,...,0,NaN,False,NaN,NaN,NaN,NaN,NaN,NM_014712.3,0
1346036,NM_005120.3(MED12):c.1540T>G (p.Cys514Gly),MED12,-1,RCV006455032,"MONDO:MONDO:0010590,MedGen:C5399762,OMIM:30545...",FG syndrome 1,GRCh38,NC_000023.11,X,71123149,...,0,Cys514Gly,False,514.0,514.0,C,C,G,NM_005120.3,1
1346037,NM_001374828.1(ARID1B):c.3584G>A (p.Trp1195Ter),ARID1B,-1,RCV006455033,"MONDO:MONDO:0007617,MedGen:C3281201,OMIM:13590...",Coffin-Siris syndrome 1,GRCh38,NC_000006.12,6,157181048,...,0,Trp1195Ter,False,1195.0,1195.0,W,W,*,NM_001374828.1,1


In [14]:
ClinVarBenchmark_VCF = Filtered_ClinVar[[
    'Chromosome', 'PositionVCF', 'VariationID', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'ClinVar_annotation', 
    'gold_stars', 'ClinVarName_refseq_ids', 'ClinVarName_start_pos', 'ClinVarName_start_aa', 'ClinVarName_alt',
    'ClinVarName_coding_sequence', 'ClinVarName_splice', 'ClinVarName_RNA_gene',
    'mRNA', 'mRNA_exon', 'coding_sequence', 'start_codon', 'stop_codon', 'five_prime_UTR', 'three_prime_UTR', 
    'mRNA_intron', 'mRNA_splice', 'snRNA', 'snRNA_exon', 'snoRNA', 'snoRNA_exon', 'other', 'transcript_set'
                                           ]]

ClinVarBenchmark_VCF = ClinVarBenchmark_VCF.rename(columns={
    'Chromosome': '#CHROM',
    'PositionVCF': 'POS',
    'VariationID': 'ID',
    'ReferenceAlleleVCF': 'REF',
    'AlternateAlleleVCF': 'ALT',
    'ClinVar_annotation': 'ClinVar_label',
    'gold_stars': 'ClinVar_gold_stars',
    'ClinVarName_start_pos': 'ClinVarName_AAPOS',
    'ClinVarName_start_aa': 'ClinVarName_AAREF', 
    'ClinVarName_alt': 'ClinVarName_AAALT',
    'mRNA': 'MANE_mRNA',
    'mRNA_exon': 'MANE_mRNA_exon',
    'coding_sequence': 'MANE_coding_sequence',
    'start_codon': 'MANE_start_codon',
    'stop_codon': 'MANE_stop_codon',
    'five_prime_UTR': 'MANE_five_prime_UTR',
    'three_prime_UTR': 'MANE_three_prime_UTR',
    'mRNA_intron': 'MANE_mRNA_intron',
    'mRNA_splice': 'MANE_mRNA_splice',
    'snRNA': 'MANE_snRNA',
    'snRNA_exon': 'MANE_snRNA_exon',
    'snoRNA': 'MANE_snoRNA',
    'snoRNA_exon': 'MANE_snoRNA_exon',
    'other': 'MANE_other',
    'transcript_set': 'MANE_transcript',
})
print(len(ClinVarBenchmark_VCF))

display(ClinVarBenchmark_VCF)

1346039


,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,MANE_five_prime_UTR,MANE_three_prime_UTR,MANE_mRNA_intron,MANE_mRNA_splice,MANE_snRNA,MANE_snRNA_exon,MANE_snoRNA,MANE_snoRNA_exon,MANE_other,MANE_transcript
0,11,126275389,5,C,T,1.0,2.0,NM_017547.4,232.0,Q,...,0,0,0,0,0,0,0,0,0,{'NM_017547.4'}
1,11,126277517,6,A,G,1.1,1.0,NM_017547.4,430.0,N,...,0,0,0,0,0,0,0,0,0,{'NM_017547.4'}
2,6,26093008,14,G,A,0.0,1.0,NM_000410.4,NaN,NaN,...,0,0,1,0,0,0,0,0,0,{'NM_000410.4'}
3,6,26092916,19,A,C,1.1,2.0,NM_000410.4,283.0,Q,...,0,0,0,0,0,0,0,0,0,{'NM_000410.4'}
4,2,19933469,23,C,T,1.1,1.0,NM_020779.4,864.0,A,...,0,0,0,0,0,0,0,0,0,"{'NM_001006657.2', 'NM_020779.4'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346034,2,120797469,4688036,G,A,0.1,1.0,NM_001374353.1,NaN,NaN,...,0,0,1,1,0,0,0,0,0,{'NM_001374353.1'}
1346035,16,30980737,4688037,A,G,0.1,1.0,NM_014712.3,NaN,NaN,...,0,0,1,1,0,0,0,0,0,{'NM_014712.3'}
1346036,X,71123149,4688038,T,G,0.1,1.0,NM_005120.3,514.0,C,...,0,0,0,0,0,0,0,0,0,{'NM_005120.3'}
1346037,6,157181048,4688039,G,A,1.0,1.0,NM_001374828.1,1195.0,W,...,0,0,0,0,0,0,0,0,0,{'NM_001374828.1'}


In [15]:
import pandas as pd
from tqdm import tqdm
from pyfaidx import Fasta
import os

# 1. 讀取並複製您需要的欄位
# column_to_preserve = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'ClinVar_label', 'ClinVar_gold_stars',
#        'ClinVarName_refseq_ids', 'ClinVarName_AAPOS', 'ClinVarName_AAREF',
#        'ClinVarName_AAALT']
df = ClinVarBenchmark_VCF.copy() #[column_to_preserve].copy()

# 2. 載入 Reference Genome
genome = Fasta(os.path.join(root_path, 'hg38', "hg38.fa"))

# 設定 window size
window = 65536
flank_size = window // 2

# 3. 定義一個「只檢查，不存序列」的驗證函數
def check_variant(row):
    try:
        # 過濾非單一鹼基變異 (Indels)
        # 如果 REF 或 ALT 長度不是 1，直接歸類為不正確 (MISMATCH)
        if pd.isna(row["REF"]) or pd.isna(row["ALT"]) or len(str(row["REF"])) != 1 or len(str(row["ALT"])) != 1:
            return 'MISMATCH_ERROR'

        chrom = str(row["#CHROM"])
        if not chrom.startswith("chr"):
            chrom = "chr" + chrom

        # 染色體不存在於 fasta 中 -> 歸類為長度/邊界錯誤
        if chrom not in genome:
            return 'LENGTH_ERROR'

        pos = int(row["POS"])
        # start = max(0, pos - flank_size - 1)
        start = pos - flank_size - 1
        end = pos + flank_size

        # 即時抽取序列 (這行執行完後，記憶體會在函數結束時自動釋放)
        seq = genome[chrom][start:end].seq.upper()

        # 檢查 1：長度是否足夠
        if len(seq) != (2 * flank_size + 1):
            return 'LENGTH_ERROR'

        # 檢查 2：中心點 Nucleotide 是否與 REF 一致
        if seq[flank_size] != str(row["REF"]):
            return 'MISMATCH_ERROR'

        # 通過所有檢查
        return 'VALID'

    except Exception as e:
        return 'LENGTH_ERROR' # 發生意外錯誤時先歸類到這裡

# 4. 啟用 tqdm apply
tqdm.pandas(desc="Checking Variants")

# 在 DataFrame 中新增一個狀態欄位 (只存極短的字串)
df['Status'] = df.progress_apply(check_variant, axis=1)

# 5. 根據 Status 切分成三個獨立的 DataFrame
df_valid = df[df['Status'] == 'VALID'].drop(columns=['Status']).copy()
df_len_error = df[df['Status'] == 'LENGTH_ERROR'].drop(columns=['Status']).copy()
df_mismatch = df[df['Status'] == 'MISMATCH_ERROR'].drop(columns=['Status']).copy()

# 6. 輸出結果報告
print("\n" + "="*40)
print(f"✅ 檢查完成！")
print(f"🟢 可順利使用 (Valid): {len(df_valid):,} 筆")
print(f"🔴 長度不足/越界 (Length Error): {len(df_len_error):,} 筆")
print(f"🟡 REF 不匹配/非 SNP (Mismatch): {len(df_mismatch):,} 筆")
print("="*40)

Checking Variants: 100%|██████████| 1346039/1346039 [08:04<00:00, 2779.54it/s]


✅ 檢查完成！
🟢 可順利使用 (Valid): 1,346,005 筆
🔴 長度不足/越界 (Length Error): 28 筆
🟡 REF 不匹配/非 SNP (Mismatch): 6 筆


In [16]:
ClinVarBenchmark_VCF=df_valid.copy().reset_index(drop=True)
ClinVarBenchmark_VCF

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,MANE_five_prime_UTR,MANE_three_prime_UTR,MANE_mRNA_intron,MANE_mRNA_splice,MANE_snRNA,MANE_snRNA_exon,MANE_snoRNA,MANE_snoRNA_exon,MANE_other,MANE_transcript
0,11,126275389,5,C,T,1.0,2.0,NM_017547.4,232.0,Q,...,0,0,0,0,0,0,0,0,0,{'NM_017547.4'}
1,11,126277517,6,A,G,1.1,1.0,NM_017547.4,430.0,N,...,0,0,0,0,0,0,0,0,0,{'NM_017547.4'}
2,6,26093008,14,G,A,0.0,1.0,NM_000410.4,NaN,NaN,...,0,0,1,0,0,0,0,0,0,{'NM_000410.4'}
3,6,26092916,19,A,C,1.1,2.0,NM_000410.4,283.0,Q,...,0,0,0,0,0,0,0,0,0,{'NM_000410.4'}
4,2,19933469,23,C,T,1.1,1.0,NM_020779.4,864.0,A,...,0,0,0,0,0,0,0,0,0,"{'NM_001006657.2', 'NM_020779.4'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346000,2,120797469,4688036,G,A,0.1,1.0,NM_001374353.1,NaN,NaN,...,0,0,1,1,0,0,0,0,0,{'NM_001374353.1'}
1346001,16,30980737,4688037,A,G,0.1,1.0,NM_014712.3,NaN,NaN,...,0,0,1,1,0,0,0,0,0,{'NM_014712.3'}
1346002,X,71123149,4688038,T,G,0.1,1.0,NM_005120.3,514.0,C,...,0,0,0,0,0,0,0,0,0,{'NM_005120.3'}
1346003,6,157181048,4688039,G,A,1.0,1.0,NM_001374828.1,1195.0,W,...,0,0,0,0,0,0,0,0,0,{'NM_001374828.1'}


In [17]:
import pandas as pd

# 假設 df_valid 已經準備好
ClinVarBenchmark_VCF = df_valid.copy().reset_index(drop=True)

# 定義我們要作為基準的 4 個欄位 (Variant 唯一識別碼)
subset_cols = ['#CHROM', 'POS', 'REF', 'ALT']

# 1. 根據這四個欄位進行排序 (Sort)
# 先排染色體，再排位置，以此類推
ClinVarBenchmark_VCF = ClinVarBenchmark_VCF.sort_values(by=subset_cols).reset_index(drop=True)

# 2. 檢查是否有完全重複的 row (基於這四個欄位)
# keep=False 代表「只要有重複，就把所有的分身都抓出來顯示」
duplicates_mask = ClinVarBenchmark_VCF.duplicated(subset=subset_cols, keep=False)
df_duplicates = ClinVarBenchmark_VCF[duplicates_mask]

# 3. 輸出結果報告
print("\n" + "="*50)
print(f"排序完成！目前資料總筆數: {len(ClinVarBenchmark_VCF):,}")

if len(df_duplicates) > 0:
    print(f"警告：發現 {len(df_duplicates):,} 筆重複的紀錄！")
    print("   (這些列在 #CHROM, POS, REF, ALT 上完全相同)")
else:
    print(f"沒有發現任何重複的變異紀錄。")
print("="*50)

# 如果有重複，可以印出前幾筆來檢查到底是哪裡不一樣 (通常是 INFO 或 ClinVar_label 不同)
if len(df_duplicates) > 0:
    display(df_duplicates.head(10))

display(ClinVarBenchmark_VCF)


排序完成！目前資料總筆數: 1,346,005
沒有發現任何重複的變異紀錄。


,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,MANE_five_prime_UTR,MANE_three_prime_UTR,MANE_mRNA_intron,MANE_mRNA_splice,MANE_snRNA,MANE_snRNA_exon,MANE_snoRNA,MANE_snoRNA_exon,MANE_other,MANE_transcript
0,1,69134,2205837,A,G,0.1,1.0,NM_001005484.2,36.0,E,...,0,0,0,0,0,0,0,0,0,{'NM_001005484.2'}
1,1,924518,3388928,G,C,0.1,1.0,NM_001385641.1,29.0,P,...,0,0,0,0,0,0,0,0,0,{'NM_001385641.1'}
2,1,925956,1543320,C,T,0.1,1.0,NM_001385641.1,184.0,I,...,0,0,0,0,0,0,0,0,0,{'NM_001385641.1'}
3,1,925969,1648427,C,T,0.1,1.0,NM_001385641.1,189.0,P,...,0,0,0,0,0,0,0,0,0,{'NM_001385641.1'}
4,1,925980,1936488,C,T,0.1,1.0,NM_001385641.1,192.0,C,...,0,0,0,0,0,0,0,0,0,{'NM_001385641.1'}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346000,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,0,0,0,0,0,0,0,0,1,set()
1346001,Y,57126292,708966,T,C,0.1,2.0,NM_005638.6,191.0,I,...,0,0,0,0,0,0,0,0,1,set()
1346002,Y,57126304,2571389,C,T,0.1,1.0,NM_005638.6,195.0,I,...,0,0,0,0,0,0,0,0,1,set()
1346003,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,0,0,0,0,0,0,0,0,1,set()


In [18]:
import itertools

amino_acid = ['G', 'Q', 'N', 'I', 'V', 'R', 'E', 'A', 'C', 'W', 'D', 'F',
       'K', 'P', 'M', 'L', 'Y', 'T', 'H', 'S']

ClinVarBenchmark_VCF['ClinVarName_missense'] = ClinVarBenchmark_VCF['ClinVarName_AAREF'].isin(amino_acid) & ClinVarBenchmark_VCF['ClinVarName_AAALT'].isin(amino_acid)
ClinVarBenchmark_VCF['ClinVarName_synonymous'] = ClinVarBenchmark_VCF['ClinVarName_AAALT'] == '='
ClinVarBenchmark_VCF['ClinVarName_stop_gain'] = ClinVarBenchmark_VCF['ClinVarName_AAALT'] == '*'
ClinVarBenchmark_VCF['Union_stop_loss'] = (ClinVarBenchmark_VCF['ClinVarName_AAREF'] == '*') | (ClinVarBenchmark_VCF['MANE_stop_codon'] == 1)
ClinVarBenchmark_VCF['Union_splice'] = (ClinVarBenchmark_VCF['MANE_mRNA_splice'] == 1) | (ClinVarBenchmark_VCF['ClinVarName_splice'] == 1)
ClinVarBenchmark_VCF['MANE_intron'] = ClinVarBenchmark_VCF['MANE_mRNA_intron'] == 1
ClinVarBenchmark_VCF['Union_RNA_gene'] = (ClinVarBenchmark_VCF['ClinVarName_RNA_gene'] == 1) | (ClinVarBenchmark_VCF['MANE_snRNA'] == 1) | (ClinVarBenchmark_VCF['MANE_snoRNA'] == 1)

conditions = [
    'MANE_start_codon', 'ClinVarName_missense', 'ClinVarName_synonymous', 'ClinVarName_stop_gain', 'Union_stop_loss',
    'MANE_five_prime_UTR', 'MANE_three_prime_UTR', 'Union_splice', 'MANE_intron', 'Union_RNA_gene'
]

pair_counts = {}

for cond1, cond2 in itertools.combinations(conditions, 2):
    count = ((ClinVarBenchmark_VCF[cond1]) & (ClinVarBenchmark_VCF[cond2])).sum()
    if count > 0:
        pair_counts[f"{cond1} + {cond2}"] = count

for k, v in pair_counts.items():
    print(f"{k}: {v}")


three_way_counts = {}

for cond1, cond2, cond3 in itertools.combinations(conditions, 3):
    overlap = ClinVarBenchmark_VCF[cond1] & ClinVarBenchmark_VCF[cond2] & ClinVarBenchmark_VCF[cond3]
    count = overlap.sum()
    if count > 0:
        key = f"{cond1} + {cond2} + {cond3}"
        three_way_counts[key] = count

for k, v in three_way_counts.items():
    print(f"{k}: {v}")

MANE_start_codon + ClinVarName_missense: 1520
MANE_start_codon + MANE_five_prime_UTR: 10
MANE_start_codon + MANE_three_prime_UTR: 4
MANE_start_codon + MANE_intron: 22
ClinVarName_missense + MANE_five_prime_UTR: 63
ClinVarName_missense + MANE_three_prime_UTR: 551
ClinVarName_missense + MANE_intron: 1848
ClinVarName_synonymous + Union_stop_loss: 406
ClinVarName_synonymous + MANE_five_prime_UTR: 362
ClinVarName_synonymous + MANE_three_prime_UTR: 2612
ClinVarName_synonymous + MANE_intron: 3543
ClinVarName_stop_gain + MANE_five_prime_UTR: 21
ClinVarName_stop_gain + MANE_three_prime_UTR: 208
ClinVarName_stop_gain + MANE_intron: 228
Union_stop_loss + MANE_three_prime_UTR: 19
Union_stop_loss + MANE_intron: 9
MANE_five_prime_UTR + MANE_three_prime_UTR: 14
MANE_five_prime_UTR + Union_splice: 10
MANE_five_prime_UTR + MANE_intron: 166
MANE_three_prime_UTR + Union_splice: 77
MANE_three_prime_UTR + MANE_intron: 1287
MANE_three_prime_UTR + Union_RNA_gene: 36
Union_splice + MANE_intron: 35852
MANE_int

In [19]:
final_subgroup = ['group: coding', 'group: noncoding',
                  'group: start loss',
                  "group: start loss + 3'UTR",
                  "group: start loss + 5'UTR",
                  'group: start loss + intron (non-splice)',
                  'group: missense',
                  "group: missense + 3'UTR",
                  "group: missense + 5'UTR",
                  'group: missense + intron (non-splice)',
                  'group: synonymous',
                  "group: synonymous + 3'UTR",
                  "group: synonymous + 5'UTR",
                  'group: synonymous + intron (non-splice)',
                  'group: stop gain',
                  "group: stop gain + 3'UTR",
                  "group: stop gain + 5'UTR",
                  'group: stop gain + intron (non-splice)',
                  'group: stop loss',
                  "group: stop loss + 3'UTR",
                  "group: stop loss + 5'UTR",
                  'group: stop synonymous',
                  "group: stop synonymous + 3'UTR",
                  "group: 5'UTR",
                  "group: 5'UTR + 3'UTR",
                  "group: 5'UTR + intron (non-splice)",
                  "group: 5'UTR + splice",
                  "group: 3'UTR",
                  "group: 3'UTR + intron (non-splice)",
                  "group: 3'UTR + splice",
                  "group: 3'UTR + RNA gene",
                  'group: splice',
                  'group: intron (non-splice)',
                  'group: intron (non-splice) + RNA gene',
                  'group: RNA gene']

In [20]:
ClinVarBenchmark_VCF[final_subgroup]=0

ClinVarBenchmark_VCF['group: coding'] = ((ClinVarBenchmark_VCF['ClinVarName_coding_sequence'] == 1) |
                                         (ClinVarBenchmark_VCF['MANE_coding_sequence'] == 1)
                                        ).astype(int)

ClinVarBenchmark_VCF['group: noncoding'] = (
                                            ~((ClinVarBenchmark_VCF['ClinVarName_coding_sequence'] == 1) |
                                              (ClinVarBenchmark_VCF['MANE_coding_sequence'] == 1))
                                        ).astype(int)

ClinVarBenchmark_VCF['group: start loss'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==1) & 
                                             (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) &
                                             (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) &
                                             (ClinVarBenchmark_VCF['MANE_intron']==0)
                                            ).astype(int)

ClinVarBenchmark_VCF["group: start loss + 3'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==1) & 
                                                     (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) &
                                                     (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) &
                                                     (ClinVarBenchmark_VCF['MANE_intron']==0)
                                                    ).astype(int)

ClinVarBenchmark_VCF["group: start loss + 5'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==1) & 
                                                     (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) &
                                                     (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) &
                                                     (ClinVarBenchmark_VCF['MANE_intron']==0)
                                                    ).astype(int)

ClinVarBenchmark_VCF['group: start loss + intron (non-splice)'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==1) & 
                                                     (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) &
                                                     (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) &
                                                     (ClinVarBenchmark_VCF['MANE_intron']==1)
                                                    ).astype(int)

ClinVarBenchmark_VCF['group: missense'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                           (ClinVarBenchmark_VCF['ClinVarName_missense']==1) & 
                                           (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) &
                                           (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) &
                                           (ClinVarBenchmark_VCF['MANE_intron']==0)
                                          ).astype(int)

ClinVarBenchmark_VCF["group: missense + 3'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                   (ClinVarBenchmark_VCF['ClinVarName_missense']==1) & 
                                                   (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                   (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                   (ClinVarBenchmark_VCF['MANE_intron']==0) 
                                                  ).astype(int)

ClinVarBenchmark_VCF["group: missense + 5'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                   (ClinVarBenchmark_VCF['ClinVarName_missense']==1) & 
                                                   (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                                   (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                   (ClinVarBenchmark_VCF['MANE_intron']==0) 
                                                  ).astype(int)

ClinVarBenchmark_VCF['group: missense + intron (non-splice)'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                                 (ClinVarBenchmark_VCF['ClinVarName_missense']==1) & 
                                                                 (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                                 (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                                 (ClinVarBenchmark_VCF['MANE_intron']==1) 
                                                                ).astype(int)

ClinVarBenchmark_VCF['group: synonymous'] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==1) & 
                                             (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) &
                                             (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                             (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                             (ClinVarBenchmark_VCF['Union_stop_loss']==0)
                                            ).astype(int)

ClinVarBenchmark_VCF["group: synonymous + 3'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==1) & 
                                                     (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) &
                                                     (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                     (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                                     (ClinVarBenchmark_VCF['Union_stop_loss']==0)
                                                    ).astype(int)

ClinVarBenchmark_VCF["group: synonymous + 5'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==1) & 
                                                     (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) &
                                                     (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                     (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                                     (ClinVarBenchmark_VCF['Union_stop_loss']==0) 
                                                    ).astype(int)

ClinVarBenchmark_VCF['group: synonymous + intron (non-splice)'] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==1) & 
                                                                 (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                                 (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                                 (ClinVarBenchmark_VCF['MANE_intron']==1) &
                                                                 (ClinVarBenchmark_VCF['Union_stop_loss']==0) 
                                                                ).astype(int)

ClinVarBenchmark_VCF['group: stop synonymous'] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==1) & 
                                                  (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                  (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                  (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                                  (ClinVarBenchmark_VCF['Union_stop_loss']==1) 
                                                 ).astype(int)

ClinVarBenchmark_VCF["group: stop synonymous + 3'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==1) & 
                                                          (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                          (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                          (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                                          (ClinVarBenchmark_VCF['Union_stop_loss']==1)
                                                          ).astype(int)

ClinVarBenchmark_VCF['group: stop gain'] = ((ClinVarBenchmark_VCF['ClinVarName_stop_gain']==1) & 
                                            (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                            (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                            (ClinVarBenchmark_VCF['MANE_intron']==0)
                                           ).astype(int)

ClinVarBenchmark_VCF["group: stop gain + 3'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_stop_gain']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                    (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_intron']==0)
                                                   ).astype(int)

ClinVarBenchmark_VCF["group: stop gain + 5'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_stop_gain']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                    (ClinVarBenchmark_VCF['MANE_intron']==0)
                                                   ).astype(int)

ClinVarBenchmark_VCF['group: stop gain + intron (non-splice)'] = ((ClinVarBenchmark_VCF['ClinVarName_stop_gain']==1) & 
                                                                  (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                                  (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                                  (ClinVarBenchmark_VCF['MANE_intron']==1)
                                                                 ).astype(int)

ClinVarBenchmark_VCF["group: stop loss"] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                            (ClinVarBenchmark_VCF['Union_stop_loss']==1) & 
                                            (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                            (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                            (ClinVarBenchmark_VCF['MANE_intron']==0)
                                           ).astype(int)

ClinVarBenchmark_VCF["group: stop loss + 3'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                    (ClinVarBenchmark_VCF['Union_stop_loss']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                    (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_intron']==0)
                                                   ).astype(int)

ClinVarBenchmark_VCF["group: stop loss + 5'UTR"] = ((ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                    (ClinVarBenchmark_VCF['Union_stop_loss']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                                    (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                    (ClinVarBenchmark_VCF['MANE_intron']==0)
                                                   ).astype(int)

ClinVarBenchmark_VCF["group: 5'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                        (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                        (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                        (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                        (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                        (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                        (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                        (ClinVarBenchmark_VCF['Union_splice']==0) &
                                        (ClinVarBenchmark_VCF['MANE_intron']==0)
                                       ).astype(int)

ClinVarBenchmark_VCF["group: 5'UTR + 3'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                                (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                (ClinVarBenchmark_VCF['Union_splice']==0) &
                                                (ClinVarBenchmark_VCF['MANE_intron']==0)
                                               ).astype(int)

ClinVarBenchmark_VCF["group: 5'UTR + intron (non-splice)"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                              (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                              (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                              (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                              (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                              (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                                              (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                              (ClinVarBenchmark_VCF['Union_splice']==0) &
                                                              (ClinVarBenchmark_VCF['MANE_intron']==1)
                                                             ).astype(int)

ClinVarBenchmark_VCF["group: 5'UTR + splice"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                 (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                 (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                 (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                 (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                 (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==1) & 
                                                 (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                 (ClinVarBenchmark_VCF['Union_splice']==1) &
                                                 (ClinVarBenchmark_VCF['MANE_intron']==1)
                                                ).astype(int)

ClinVarBenchmark_VCF["group: 3'UTR"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                        (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                        (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                        (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                        (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                        (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                        (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                        (ClinVarBenchmark_VCF['Union_splice']==0) &
                                        (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                        (ClinVarBenchmark_VCF['Union_RNA_gene']==0)
                                       ).astype(int)

ClinVarBenchmark_VCF["group: 3'UTR + intron (non-splice)"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                              (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                              (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                              (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                              (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                              (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                              (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                              (ClinVarBenchmark_VCF['Union_splice']==0) &
                                                              (ClinVarBenchmark_VCF['MANE_intron']==1) &
                                                              (ClinVarBenchmark_VCF['Union_RNA_gene']==0)
                                                             ).astype(int)

ClinVarBenchmark_VCF["group: 3'UTR + splice"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                 (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                 (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                 (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                 (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                 (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                 (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                 (ClinVarBenchmark_VCF['Union_splice']==1) &
                                                 (ClinVarBenchmark_VCF['MANE_intron']==1) &
                                                 (ClinVarBenchmark_VCF['Union_RNA_gene']==0)
                                                 ).astype(int)

ClinVarBenchmark_VCF["group: 3'UTR + RNA gene"] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                   (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                   (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                   (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                   (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                   (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                   (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==1) & 
                                                   (ClinVarBenchmark_VCF['Union_splice']==0) &
                                                   (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                                   (ClinVarBenchmark_VCF['Union_RNA_gene']==1)
                                                   ).astype(int)

ClinVarBenchmark_VCF['group: splice'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                         (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                         (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                         (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                         (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                         (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                         (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                         (ClinVarBenchmark_VCF['Union_splice']==1) &
                                         (ClinVarBenchmark_VCF['MANE_intron']==1) &
                                         (ClinVarBenchmark_VCF['Union_RNA_gene']==0)
                                        ).astype(int)

ClinVarBenchmark_VCF['group: intron (non-splice)'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                      (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                      (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                      (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                      (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                      (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                      (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                      (ClinVarBenchmark_VCF['Union_splice']==0) &
                                                      (ClinVarBenchmark_VCF['MANE_intron']==1) &
                                                      (ClinVarBenchmark_VCF['Union_RNA_gene']==0)
                                                     ).astype(int)

ClinVarBenchmark_VCF['group: intron (non-splice) + RNA gene'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                                                  (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                                                  (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                                                  (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                                                  (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                                                  (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                                                  (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                                                  (ClinVarBenchmark_VCF['Union_splice']==0) &
                                                                  (ClinVarBenchmark_VCF['MANE_intron']==1) &
                                                                  (ClinVarBenchmark_VCF['Union_RNA_gene']==1)
                                                                 ).astype(int)


ClinVarBenchmark_VCF['group: RNA gene'] = ((ClinVarBenchmark_VCF['MANE_start_codon']==0) & 
                                          (ClinVarBenchmark_VCF['ClinVarName_missense']==0) & 
                                          (ClinVarBenchmark_VCF['ClinVarName_synonymous']==0) & 
                                          (ClinVarBenchmark_VCF['ClinVarName_stop_gain']==0) & 
                                          (ClinVarBenchmark_VCF['Union_stop_loss']==0) & 
                                          (ClinVarBenchmark_VCF['MANE_five_prime_UTR']==0) & 
                                          (ClinVarBenchmark_VCF['MANE_three_prime_UTR']==0) & 
                                          (ClinVarBenchmark_VCF['Union_splice']==0) &
                                          (ClinVarBenchmark_VCF['MANE_intron']==0) &
                                          (ClinVarBenchmark_VCF['Union_RNA_gene']==1)
                                         ).astype(int)

In [21]:
result = {}

for col in final_subgroup:
    subset = ClinVarBenchmark_VCF[ClinVarBenchmark_VCF[col] == 1]
    pathogenic_count = (subset['ClinVar_label'] == 1).sum()
    benign_count = (subset['ClinVar_label'] == 0).sum()
    all_count = pathogenic_count + benign_count
    result[col] = {'pathogenic': pathogenic_count, 'benign': benign_count, 'total': all_count}

import pandas as pd
summary_df = pd.DataFrame(result).T
display(summary_df)
# summary_df.to_csv('ClinVarBenchmark_PB_202504_groupcount.csv')

,pathogenic,benign,total
group: coding,58229,67702,125931
group: noncoding,11286,104915,116201
group: start loss,772,19,791
group: start loss + 3'UTR,2,0,2
group: start loss + 5'UTR,4,0,4
group: start loss + intron (non-splice),13,2,15
group: missense,15185,27273,42458
group: missense + 3'UTR,47,53,100
group: missense + 5'UTR,0,9,9
group: missense + intron (non-splice),78,224,302


In [22]:
print('MANE', sum(ClinVarBenchmark_VCF['MANE_coding_sequence']==1))
print('ClinVar', sum(ClinVarBenchmark_VCF['ClinVarName_coding_sequence']==1))
print('Both', sum((ClinVarBenchmark_VCF['MANE_coding_sequence']==1) & (ClinVarBenchmark_VCF['ClinVarName_coding_sequence']==1)))

MANE 880633
ClinVar 882387
Both 880255


In [23]:
import pandas as pd
import numpy as np

# 1. 讀取 UCSC 的 NCBI RefSeq 資料庫 (基於人類 hg38)
print("正在從 UCSC 獲取 RefSeq 資料庫 (可能需要幾十秒下載)...")
url = "http://hgdownload.cse.ucsc.edu/goldenPath/hg38/database/ncbiRefSeq.txt.gz"

# 根據 UCSC Table Schema: 第 2 欄 (index 1) 是 name (NM_ID)，第 4 欄 (index 3) 是 strand (+/-)
refseq_df = pd.read_csv(url, sep='\t', header=None, usecols=[1, 3], names=['refseq_id', 'strand'])

# 2. 建立精確匹配字典 (包含版號，如 NM_003372.7 -> +)
refseq_dict_exact = refseq_df.drop_duplicates(subset=['refseq_id']).set_index('refseq_id')['strand'].to_dict()

# 3. 建立模糊匹配字典 (去除版號，如 NM_003372 -> +) 作為備用防護網
refseq_df['refseq_id_base'] = refseq_df['refseq_id'].str.split('.').str[0]
refseq_dict_base = refseq_df.drop_duplicates(subset=['refseq_id_base']).set_index('refseq_id_base')['strand'].to_dict()

print(f"✅ 字典建立完成！共載入 {len(refseq_dict_exact)} 筆 Transcript 資訊。")

# 4. 定義 Mapping 函數
def get_strand(refseq_id):
    if pd.isna(refseq_id):
        return np.nan
    
    refseq_id = str(refseq_id).strip()
    
    # 嘗試 1: 完全匹配
    if refseq_id in refseq_dict_exact:
        return refseq_dict_exact[refseq_id]
    
    # 嘗試 2: 略過版號匹配 (切掉小數點後面的數字)
    base_id = refseq_id.split('.')[0]
    return refseq_dict_base.get(base_id, np.nan)

# 5. 映射到您的 clinvar DataFrame
print("正在將 Strand 資訊合併進 clinvar...")
ClinVarBenchmark_VCF['Strand'] = ClinVarBenchmark_VCF['ClinVarName_refseq_ids'].apply(get_strand)

# 檢查 mapping 結果
total_rows = len(ClinVarBenchmark_VCF)
missing_rows = ClinVarBenchmark_VCF['Strand'].isna().sum()
success_rate = ((total_rows - missing_rows) / total_rows) * 100

print(f"\n🎉 Mapping 完成！成功率: {success_rate:.2f}% (找不到的筆數: {missing_rows})")

# 顯示前幾筆資料供您確認
display(ClinVarBenchmark_VCF[['ClinVarName_refseq_ids', 'Strand']].head(10))

正在從 UCSC 獲取 RefSeq 資料庫 (可能需要幾十秒下載)...
✅ 字典建立完成！共載入 186198 筆 Transcript 資訊。
正在將 Strand 資訊合併進 clinvar...

🎉 Mapping 完成！成功率: 99.59% (找不到的筆數: 5576)


,ClinVarName_refseq_ids,Strand
0,NM_001005484.2,+
1,NM_001385641.1,+
2,NM_001385641.1,+
3,NM_001385641.1,+
4,NM_001385641.1,+
5,NM_001385641.1,+
6,NM_001385641.1,+
7,NM_001385641.1,+
8,NM_001385641.1,+
9,NM_001385641.1,+


In [24]:
# 將布林值 (True/False) 直接轉換為整數 (1/0)
ClinVarBenchmark_VCF['group: +'] = (ClinVarBenchmark_VCF['Strand'] == '+').astype(int)
ClinVarBenchmark_VCF['group: -'] = (ClinVarBenchmark_VCF['Strand'] == '-').astype(int)

# 檢查一下轉換結果
print("各股數量的統計：")
print(f"group: + (正股) 的數量: {ClinVarBenchmark_VCF['group: +'].sum():,}")
print(f"group: - (反股) 的數量: {ClinVarBenchmark_VCF['group: -'].sum():,}")
print(f"找不到 Strand 的數量: {ClinVarBenchmark_VCF['Strand'].isna().sum():,}")

# 顯示前幾筆資料，確認 One-hot encoding 的結果
display(ClinVarBenchmark_VCF[['ClinVarName_refseq_ids', 'Strand', 'group: +', 'group: -']].head(10))

各股數量的統計：
group: + (正股) 的數量: 662,655
group: - (反股) 的數量: 677,774
找不到 Strand 的數量: 5,576


,ClinVarName_refseq_ids,Strand,group: +,group: -
0,NM_001005484.2,+,1,0
1,NM_001385641.1,+,1,0
2,NM_001385641.1,+,1,0
3,NM_001385641.1,+,1,0
4,NM_001385641.1,+,1,0
5,NM_001385641.1,+,1,0
6,NM_001385641.1,+,1,0
7,NM_001385641.1,+,1,0
8,NM_001385641.1,+,1,0
9,NM_001385641.1,+,1,0


In [ ]:
ClinVarBenchmark_VCF.to_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PLPBLB_202602.csv'), index=False)

# Making final benchmark

## Full VCF

In [25]:
import pandas as pd
import os

ClinVarBenchmark_VCF= pd.read_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PLPBLB_202602.csv'))
ClinVarBenchmark_VCF

/tmp/ipykernel_2383436/700037280.py:4: DtypeWarning: Columns (0: #CHROM) have mixed types. Specify dtype option on import or set low_memory=False.
  ClinVarBenchmark_VCF= pd.read_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PLPBLB_202602.csv'))


,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,group: 3'UTR + intron (non-splice),group: 3'UTR + splice,group: 3'UTR + RNA gene,group: splice,group: intron (non-splice),group: intron (non-splice) + RNA gene,group: RNA gene,Strand,group: +,group: -
0,1,69134,2205837,A,G,0.1,1.0,NM_001005484.2,36.0,E,...,0,0,0,0,0,0,0,+,1,0
1,1,924518,3388928,G,C,0.1,1.0,NM_001385641.1,29.0,P,...,0,0,0,0,0,0,0,+,1,0
2,1,925956,1543320,C,T,0.1,1.0,NM_001385641.1,184.0,I,...,0,0,0,0,0,0,0,+,1,0
3,1,925969,1648427,C,T,0.1,1.0,NM_001385641.1,189.0,P,...,0,0,0,0,0,0,0,+,1,0
4,1,925980,1936488,C,T,0.1,1.0,NM_001385641.1,192.0,C,...,0,0,0,0,0,0,0,+,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346000,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,0,0,0,0,0,0,0,+,1,0
1346001,Y,57126292,708966,T,C,0.1,2.0,NM_005638.6,191.0,I,...,0,0,0,0,0,0,0,+,1,0
1346002,Y,57126304,2571389,C,T,0.1,1.0,NM_005638.6,195.0,I,...,0,0,0,0,0,0,0,+,1,0
1346003,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,0,0,0,0,0,0,0,+,1,0


In [26]:
ClinVarBenchmark_VCF = ClinVarBenchmark_VCF[['#CHROM', 'POS', 'REF', 'ALT', 'ClinVar_label']].copy()
ClinVarBenchmark_VCF.to_csv(os.path.join(root_path, 'data', 'ClinVarVCF_202602.csv'), index=False)

## PB benchmark

In [7]:
import pandas as pd
import os

ClinVarBenchmark_VCF= pd.read_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PLPBLB_202602.csv'))
ClinVarBenchmark_VCF

/tmp/ipykernel_2526539/700037280.py:4: DtypeWarning: Columns (0: #CHROM) have mixed types. Specify dtype option on import or set low_memory=False.
  ClinVarBenchmark_VCF= pd.read_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PLPBLB_202602.csv'))


,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,group: 3'UTR + intron (non-splice),group: 3'UTR + splice,group: 3'UTR + RNA gene,group: splice,group: intron (non-splice),group: intron (non-splice) + RNA gene,group: RNA gene,Strand,group: +,group: -
0,1,69134,2205837,A,G,0.1,1.0,NM_001005484.2,36.0,E,...,0,0,0,0,0,0,0,+,1,0
1,1,924518,3388928,G,C,0.1,1.0,NM_001385641.1,29.0,P,...,0,0,0,0,0,0,0,+,1,0
2,1,925956,1543320,C,T,0.1,1.0,NM_001385641.1,184.0,I,...,0,0,0,0,0,0,0,+,1,0
3,1,925969,1648427,C,T,0.1,1.0,NM_001385641.1,189.0,P,...,0,0,0,0,0,0,0,+,1,0
4,1,925980,1936488,C,T,0.1,1.0,NM_001385641.1,192.0,C,...,0,0,0,0,0,0,0,+,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346000,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,0,0,0,0,0,0,0,+,1,0
1346001,Y,57126292,708966,T,C,0.1,2.0,NM_005638.6,191.0,I,...,0,0,0,0,0,0,0,+,1,0
1346002,Y,57126304,2571389,C,T,0.1,1.0,NM_005638.6,195.0,I,...,0,0,0,0,0,0,0,+,1,0
1346003,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,0,0,0,0,0,0,0,+,1,0


In [8]:
ClinVarBenchmark_VCF_PB = ClinVarBenchmark_VCF[ClinVarBenchmark_VCF['ClinVar_label'].isin([0, 1])].reset_index(drop=True)
ClinVarBenchmark_VCF_PB.to_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PB_202602.csv'), index=False)
ClinVarBenchmark_VCF_PB

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,group: 3'UTR + intron (non-splice),group: 3'UTR + splice,group: 3'UTR + RNA gene,group: splice,group: intron (non-splice),group: intron (non-splice) + RNA gene,group: RNA gene,Strand,group: +,group: -
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,0,0,0,0,0,0,0,+,1,0
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,0,0,0,0,0,0,0,+,1,0
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,0,0,0,0,0,0,0,+,1,0
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,0,0,0,0,0,0,0,+,1,0
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,0,0,0,0,0,0,0,+,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,Y,56960499,726033,A,G,0.0,1.0,NM_001304990.2,36.0,K,...,0,0,0,0,0,0,0,+,1,0
242128,Y,57084754,777848,C,G,0.0,2.0,NM_005638.6,109.0,V,...,0,0,0,0,0,0,0,+,1,0
242129,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,0,0,0,0,0,0,0,+,1,0
242130,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,0,0,0,0,0,0,0,+,1,0


In [29]:
results= pd.read_csv("./clinvar_updated.csv")
results

/tmp/ipykernel_2383436/3712172788.py:1: DtypeWarning: Columns (0: #CHROM) have mixed types. Specify dtype option on import or set low_memory=False.
  results= pd.read_csv("./clinvar_updated.csv")


,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloP,GPN_MSA,ESM1b,ESM1v,ESM2,PrimateAI_3D,AlphaMissense,PhyloGPN,Evo2_40B,Rule_based
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,5.113,-5.23,-13.005167,-6.148792,-4.594111,NaN,NaN,-1.801111,-0.001966,0.357648
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,3.430,-2.89,-9.364201,-5.213579,-4.350942,NaN,NaN,-0.810768,-0.001260,0.357648
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,0.322,-1.54,-4.432576,-4.698092,-4.863894,NaN,NaN,-0.815928,0.000659,0.357648
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,3.747,-4.98,-4.216251,-1.332169,-3.825120,NaN,NaN,-1.633337,-0.003418,0.357648
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,4.888,-6.17,NaN,NaN,NaN,NaN,NaN,-1.464217,-0.000417,0.005252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,X,154999538,3907432,A,G,1.0,1.0,NM_000132.4,69.0,L,...,6.621,-7.56,-9.788183,-5.935312,-8.724833,0.695317,0.7201,-3.229128,-0.005185,0.357648
242128,X,155022455,4081666,C,T,1.0,1.0,NM_000132.4,33.0,W,...,3.502,-7.16,-23.068070,-17.417273,-20.274853,0.960201,0.9986,-6.841917,-0.033375,0.996788
242129,Y,634740,3905571,C,T,1.0,1.0,NM_000451.4,134.0,R,...,1.163,NaN,-24.036531,-24.916836,-18.373589,0.981721,1.0000,-4.706703,-0.001884,0.996788
242130,Y,1299043,4278984,C,T,0.0,1.0,NM_172245.4,NaN,NaN,...,-0.251,NaN,NaN,NaN,NaN,NaN,NaN,-2.070815,-0.000091,0.020306


In [32]:
for column, count in results.isna().sum().items():
    print(f"{column}: {count}")

#CHROM: 0
POS: 0
ID: 0
REF: 0
ALT: 0
ClinVar_label: 0
ClinVar_gold_stars: 0
ClinVarName_refseq_ids: 2897
ClinVarName_AAPOS: 116359
ClinVarName_AAREF: 116359
ClinVarName_AAALT: 116359
ClinVarName_coding_sequence: 0
ClinVarName_splice: 0
ClinVarName_RNA_gene: 0
MANE_mRNA: 0
MANE_mRNA_exon: 0
MANE_coding_sequence: 0
MANE_start_codon: 0
MANE_stop_codon: 0
MANE_five_prime_UTR: 0
MANE_three_prime_UTR: 0
MANE_mRNA_intron: 0
MANE_mRNA_splice: 0
MANE_snRNA: 0
MANE_snRNA_exon: 0
MANE_snoRNA: 0
MANE_snoRNA_exon: 0
MANE_other: 0
MANE_transcript: 0
ClinVarName_missense: 0
ClinVarName_synonymous: 0
ClinVarName_stop_gain: 0
Union_stop_loss: 0
Union_splice: 0
MANE_intron: 0
Union_RNA_gene: 0
group: coding: 0
group: noncoding: 0
group: start loss: 0
group: start loss + 3'UTR: 0
group: start loss + 5'UTR: 0
group: start loss + intron (non-splice): 0
group: missense: 0
group: missense + 3'UTR: 0
group: missense + 5'UTR: 0
group: missense + intron (non-splice): 0
group: synonymous: 0
group: synonymous + 3

In [35]:
results[results.GPN_MSA.isna()]

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloP,GPN_MSA,ESM1b,ESM1v,ESM2,PrimateAI_3D,AlphaMissense,PhyloGPN,Evo2_40B,Rule_based
791,1,5875102,167375,T,T,0.0,2.0,NM_015102.5,NaN,NaN,...,5.334,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.992923
2038,1,11794400,515933,G,G,0.0,1.0,NM_005957.5,435.0,F,...,0.648,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.005252
2431,1,12358644,1227847,C,C,0.0,2.0,NM_015378.4,NaN,NaN,...,-2.299,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.011909
2450,1,12507178,1249759,G,G,0.0,2.0,NM_015378.4,NaN,NaN,...,-1.306,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.011909
3306,1,20661380,540210,G,G,0.0,1.0,NaN,NaN,NaN,...,-0.184,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.020306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235318,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,2.288,NaN,-5.577450,-0.748903,-5.272952,0.300756,0.08,-3.095034,-0.000956,0.357648
235319,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,-0.570,NaN,NaN,NaN,NaN,NaN,NaN,-0.255677,-0.000129,0.020306
235320,Y,57191079,783917,C,T,0.0,2.0,NM_002186.3,191.0,A,...,0.576,NaN,NaN,NaN,NaN,NaN,NaN,-1.238647,-0.000694,0.005252
242129,Y,634740,3905571,C,T,1.0,1.0,NM_000451.4,134.0,R,...,1.163,NaN,-24.036531,-24.916836,-18.373589,0.981721,1.00,-4.706703,-0.001884,0.996788


In [ ]:
ClinVarBenchmark_VCF= pd.read_csv(os.path.join(root_path, 'data', 'ClinVarVCF_202602.csv'))
ClinVarBenchmark_VCF

In [ ]:
ClinVarBenchmark_VCF_PB = ClinVarBenchmark_VCF[ClinVarBenchmark_VCF['ClinVar_label'].isin([0, 1])].reset_index(drop=True)
ClinVarBenchmark_VCF_PB.to_csv(os.path.join(root_path, 'data', 'ClinVarVCF_PB_202602.csv'), index=False)
ClinVarBenchmark_VCF_PB

## Supplementary Benchmark

In [ ]:
import pandas as pd
import os

ClinVarBenchmark_VCF= pd.read_csv(os.path.join(root_path, 'data', 'ClinVarBenchmark_PLPBLB_202602.csv'))
ClinVarBenchmark_VCF

In [ ]:
import pandas as pd

# 定義 label 對應關係
label_map = {0: 'Benign', 0.1: 'Likely Benign', 1: 'Pathogenic', 1.1: 'Likely Pathogenic'}

# 找出所有 group: 開頭的欄位
group_cols = [col for col in ClinVarBenchmark_VCF.columns if col.startswith('group:')]

# 建立結果表格
rows = []
for col in group_cols:
    subset = ClinVarBenchmark_VCF[ClinVarBenchmark_VCF[col] == 1]
    counts = subset['ClinVar_label'].value_counts()
    row = {'Group': col.replace('group: ', '')}
    for label_val, label_name in label_map.items():
        row[label_name] = counts.get(label_val, 0)
    row['Total'] = sum(row[name] for name in label_map.values())
    rows.append(row)

result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False))

In [ ]:
target_groups = [
    "coding", "noncoding",
    "intron (non-splice)", "splice", "5'UTR", "3'UTR", "RNA gene",
    "synonymous", "start loss", "stop loss", "stop gain", "missense",
    "missense + intron (non-splice)", "missense + 3'UTR"
]

# 過濾 result_df 中符合 target_groups 的列
filtered_df = result_df[result_df['Group'].isin(target_groups)].copy()

# 找出 Pathogenic 或 Benign 不足 100 的列
mask = (filtered_df['Pathogenic'] < 100) | (filtered_df['Benign'] < 100)
low_count_df = filtered_df[mask]

print(low_count_df.to_string(index=False))

In [ ]:
import os

# 對應 class_to_add 到原始欄位名稱
class_to_add = [
    'start loss', "missense + 3'UTR", 'missense + intron (non-splice)',
    'stop loss', "5'UTR", "3'UTR", "splice", "RNA gene"
]
group_cols_filter = ['group: ' + c for c in class_to_add]

# 篩選：至少一個 group 欄位為 1，且 ClinVar_label 為 0.1 或 1.1
mask_group = ClinVarBenchmark_VCF[group_cols_filter].eq(1).any(axis=1)
mask_label = ClinVarBenchmark_VCF['ClinVar_label'].isin([0.1, 1.1])

supplement_df = ClinVarBenchmark_VCF[mask_group & mask_label].copy()

# 整理欄位並存檔
supplement_out = supplement_df[['#CHROM', 'POS', 'REF', 'ALT', 'ClinVar_label']].copy()
supplement_out.to_csv(os.path.join(root_path, 'data', 'ClinVarVCF_supplement_202602.csv'), index=False)

print(f"篩選出 {len(supplement_out)} 筆資料")
print(supplement_out['ClinVar_label'].value_counts())

# 2023.10 benchmark

In [1]:
import os
import pandas as pd

ClinVar = pd.read_csv("./variant_summary_2023-10.txt.gz", 
                      sep='\t', dtype={'Chromosome': str}, low_memory=False, compression='gzip') 

display(ClinVar)

# for col in ClinVar.columns:
#     print(col)
#     print(ClinVar[col].unique())
#     print()

ClinVar = ClinVar[ClinVar['Assembly'] == 'GRCh38']
print('GRCh38', ClinVar.shape[0])

# ClinVar = ClinVar[ClinVar['OriginSimple']!='somatic'].reset_index(drop=True)
# print('Exclude somatic', len(ClinVar), 'remain')

basic_chromosome = list(map(str, range(1,23))) + ['X', 'Y']
ClinVar = ClinVar[ClinVar['Chromosome'].isin(basic_chromosome)]
    
print('Basic Chromosome', ClinVar.shape[0])

for mut in ClinVar['Type'].unique():
    print(mut, ClinVar[ClinVar['Type']==mut].shape[0])

ClinVar = ClinVar[ClinVar['Type'] == 'single nucleotide variant']
print('single nucleotide variant', ClinVar.shape[0])

# import numpy as np

# REVIEW_STATUS_TO_GOLD_STARS = {
#     'criteria provided, single submitter': 1,
#     'criteria provided, multiple submitters, no conflicts': 2,
#     'criteria provided, conflicting interpretations': 1,
#     'no assertion criteria provided': np.nan,
#     'reviewed by expert panel': 3,
#     'no assertion provided': np.nan,
#     'no interpretation for the single variant': np.nan,
#     'practice guideline': 4,
#     }

# CLINICAL_SIGNIFICANCE_TO_LABEL = {
#     'Benign': 0,
#     'Likely benign': 0.1,
#     'Benign/Likely benign': 0.1,
#     'Pathogenic': 1,
#     'Likely pathogenic': 1.1,
#     'Pathogenic/Likely pathogenic': 1.1,
#     'Uncertain significance': 2
#     }

# ClinVar['ClinVar_annotation'] = ClinVar['ClinicalSignificance'].map(CLINICAL_SIGNIFICANCE_TO_LABEL)
# ClinVar = ClinVar[ClinVar['ClinVar_annotation'].isin({0, 0.1, 1, 1.1})]

# ClinVar['gold_stars'] = ClinVar['ReviewStatus'].map(REVIEW_STATUS_TO_GOLD_STARS)
# print('Total Number:\t', len(ClinVar))
# print('Benign:\t\t', len(ClinVar[ClinVar['ClinVar_annotation']==0]))
# print('Likely Benign:\t\t', len(ClinVar[ClinVar['ClinVar_annotation']==0.1]))
# print('Pathogenic:\t', len(ClinVar[ClinVar['ClinVar_annotation']==1]))
# print('Likely Pathogenic:\t', len(ClinVar[ClinVar['ClinVar_annotation']==1.1]))
# # print('VUS:\t\t', len(ClinVar[ClinVar['ClinVar_annotation']==2]))

# ClinVar = ClinVar[ClinVar['gold_stars'].isin({1, 2, 3, 4})]
# print('Gold Stars >=1 :', len(ClinVar))

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic,1,-,397704705,...,"criteria provided, single submitter",2,-,N,"ClinGen:CA215070,OMIM:613653.0001",3,2,4820844,GGAT,TGCTGTAAACTGTAACTGTAAA
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic,1,-,397704705,...,"criteria provided, single submitter",2,-,N,"ClinGen:CA215070,OMIM:613653.0001",3,2,4781213,GGAT,TGCTGTAAACTGTAACTGTAAA
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,no assertion criteria provided,1,-,N,"ClinGen:CA215072,OMIM:613653.0002",1,3,4827360,GCTGCTGGACCTGCC,G
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,no assertion criteria provided,1,-,N,"ClinGen:CA215072,OMIM:613653.0002",1,3,4787729,GCTGCTGGACCTGCC,G
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,no assertion criteria provided,1,-,N,"ClinGen:CA210674,UniProtKB:Q92610#VAR_064583,O...",1,4,85342440,G,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4616361,2748658,Deletion,NM_004247.4(EFTUD2):c.2643_2649del (p.Phe881fs),9343,EFTUD2,HGNC:30858,Pathogenic,1,"Nov 15, 2019",-1,...,"criteria provided, single submitter",1,-,N,-,2,2581074,44852474,CAAAGCCA,C
4616362,2748659,Insertion,NM_001042492.3(NF1):c.1019_1020insGA (p.Val341fs),4763,NF1,HGNC:7765,Pathogenic,1,"Apr 10, 2020",-1,...,"criteria provided, single submitter",1,-,N,-,2,2581075,29527570,C,CGA
4616363,2748659,Insertion,NM_001042492.3(NF1):c.1019_1020insGA (p.Val341fs),4763,NF1,HGNC:7765,Pathogenic,1,"Apr 10, 2020",-1,...,"criteria provided, single submitter",1,-,N,-,2,2581075,31200552,C,CGA
4616364,2748660,Deletion,NM_001352027.3(PHF21A):c.19del (p.Gln7fs),51317,PHF21A,HGNC:24156,Likely pathogenic,1,"May 02, 2023",-1,...,"criteria provided, single submitter",1,-,N,-,2,2581076,46105751,TG,T


GRCh38 2274674
Basic Chromosome 2271766
Indel 11838
Deletion 100831
single nucleotide variant 2061181
Duplication 45895
Microsatellite 27492
Insertion 9572
Translocation 24
Inversion 1070
copy number gain 6883
Variation 345
copy number loss 6632
Complex 3
single nucleotide variant 2061181


In [2]:
ClinVar = ClinVar.rename(columns={
    'Chromosome': '#CHROM',
    'PositionVCF': 'POS',
    'VariationID': 'ID',
    'ReferenceAlleleVCF': 'REF',
    'AlternateAlleleVCF': 'ALT',
    'ClinVar_annotation': 'ClinVar_label',
    'gold_stars': 'ClinVar_gold_stars',
    'ClinVarName_start_pos': 'ClinVarName_AAPOS',
    'ClinVarName_start_aa': 'ClinVarName_AAREF', 
    'ClinVarName_alt': 'ClinVarName_AAALT',
    'mRNA': 'MANE_mRNA',
    'mRNA_exon': 'MANE_mRNA_exon',
    'coding_sequence': 'MANE_coding_sequence',
    'start_codon': 'MANE_start_codon',
    'stop_codon': 'MANE_stop_codon',
    'five_prime_UTR': 'MANE_five_prime_UTR',
    'three_prime_UTR': 'MANE_three_prime_UTR',
    'mRNA_intron': 'MANE_mRNA_intron',
    'mRNA_splice': 'MANE_mRNA_splice',
    'snRNA': 'MANE_snRNA',
    'snRNA_exon': 'MANE_snRNA_exon',
    'snoRNA': 'MANE_snoRNA',
    'snoRNA_exon': 'MANE_snoRNA_exon',
    'other': 'MANE_other',
    'transcript_set': 'MANE_transcript',
})

In [3]:
import pandas as pd

ClinVarBenchmark_VCF_PB = pd.read_csv("./clinvar_updated.csv", low_memory=False)
ClinVarBenchmark_VCF_PB

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max,vesm_score,gpnstar_v_llr
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,-1.801111,-0.001966,0.357648,-1.033955,-1.668930e-06,-1.666055,0.000043,2.827893,NaN,-5.484859
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,-0.810768,-0.001260,0.357648,-1.569786,4.708767e-06,-1.588054,0.000701,3.240415,NaN,-3.155202
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,-0.815928,0.000659,0.357648,-1.588131,3.576279e-07,-1.680781,0.000585,3.104318,NaN,-1.901591
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,-1.633337,-0.003418,0.357648,-3.150200,7.659197e-06,-3.216790,-0.000197,1.822163,NaN,-4.783436
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,-1.464217,-0.000417,0.005252,-4.653107,-1.162291e-05,-3.354657,0.000510,2.667778,NaN,-4.984299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,Y,56960499,726033,A,G,0.0,1.0,NM_001304990.2,36.0,K,...,-0.414166,-0.000091,0.005252,-1.494453,-5.513430e-06,-2.435236,-0.001451,2.540560,NaN,-2.214205
242128,Y,57084754,777848,C,G,0.0,2.0,NM_005638.6,109.0,V,...,-1.074717,-0.000446,0.005252,-1.905708,-1.013279e-06,-1.233571,-0.002534,2.924839,NaN,-4.122275
242129,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,-3.095034,-0.000956,0.357648,-6.131863,-4.357100e-05,-2.936291,-0.000638,2.801300,-3.9775,-3.547984
242130,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,-0.255677,-0.000129,0.020306,-2.886108,5.185604e-06,0.014036,-0.005069,2.687465,NaN,-2.359769


In [4]:
ClinVar[['#CHROM', 'POS', 'ID', 'REF', 'ALT']]

,#CHROM,POS,ID,REF,ALT
5,15,84799209,4,G,A
7,11,126275389,5,C,T
9,11,126277517,6,A,G
11,14,31562125,214885,G,A
13,6,26092913,9,G,A
...,...,...,...,...,...
4616345,5,61521369,2581066,G,A
4616349,2,178548955,2581068,C,A
4616351,3,146071400,2581069,A,C
4616353,1,152309872,2581070,G,A


In [5]:
keys = ['#CHROM', 'POS', 'ID', 'REF', 'ALT']

# Sort 兩個 dataframe（原地或新建都可以）
ClinVarBenchmark_VCF_PB = ClinVarBenchmark_VCF_PB.sort_values(keys).reset_index(drop=True)
ClinVar_sorted = ClinVar[keys].sort_values(keys).reset_index(drop=True)

# Left join，indicator 標記哪些 row 有在 ClinVar 出現
merged = ClinVarBenchmark_VCF_PB.merge(
    ClinVar_sorted.drop_duplicates(subset=keys),
    on=keys,
    how='left',
    indicator=True
)

# 只保留「左邊獨有」的 row（即不在 ClinVar 中的）
result = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
result

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max,vesm_score,gpnstar_v_llr
8,1,935817,3910045,C,T,0.0,1.0,NM_001385641.1,296.0,S,...,-2.058084,-0.003259,0.005252,-3.855202,-0.000014,-3.870307,-0.000138,2.941760,NaN,-4.067893
61,1,963087,2920761,C,T,0.0,1.0,NM_198317.3,NaN,NaN,...,0.415658,0.000355,0.011909,-0.374943,0.000008,-2.267148,0.000046,2.759861,NaN,0.075307
80,1,1014545,2637890,C,T,0.0,1.0,NaN,NaN,NaN,...,0.049672,-0.000090,0.020306,-0.592770,0.000002,-2.546340,0.001286,2.598526,NaN,1.192076
87,1,1022368,3255595,C,A,1.0,1.0,NM_198576.4,123.0,Y,...,-7.469908,-0.001843,0.996788,-6.607964,-0.000052,-2.148357,-0.002472,2.863250,-17.5720,-10.120655
96,1,1041048,3910046,C,A,0.0,1.0,NM_198576.4,NaN,NaN,...,-2.010926,-0.000613,0.011909,-4.626171,-0.000020,-0.821669,-0.000096,2.969108,NaN,-1.131363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242072,Y,1299043,4278984,C,T,0.0,1.0,NM_172245.4,NaN,NaN,...,-2.070815,-0.000091,0.020306,-3.350967,-0.000004,-2.009600,0.000669,1.950964,NaN,-1.859038
242096,Y,1309528,3036043,G,A,0.0,1.0,NM_172245.4,NaN,NaN,...,0.589955,0.000165,0.020306,-0.742752,0.000001,-0.518977,-0.005532,2.453441,NaN,-0.059930
242120,Y,2787425,3338847,A,G,1.0,1.0,NM_003140.3,60.0,V,...,-0.207253,-0.001519,0.357648,-6.535221,-0.000037,-0.822804,-0.011490,2.093392,-7.1130,-5.224675
242123,Y,2787459,2829588,C,A,1.0,1.0,NM_003140.3,49.0,G,...,-2.273096,-0.006388,0.996788,-4.992516,-0.000030,-2.166879,0.015845,2.924799,-19.4554,-6.337333


In [6]:
result.to_csv("./clinvar_2023.csv", index=False)

# balanced benchmark

In [1]:
import pandas as pd

ClinVarBenchmark_VCF_PB = pd.read_csv("./clinvar_updated.csv", low_memory=False)
ClinVarBenchmark_VCF_PB

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max,vesm_score,gpnstar_v_llr
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,-1.801111,-0.001966,0.357648,-1.033955,-1.668930e-06,-1.666055,0.000043,2.827893,NaN,-5.484859
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,-0.810768,-0.001260,0.357648,-1.569786,4.708767e-06,-1.588054,0.000701,3.240415,NaN,-3.155202
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,-0.815928,0.000659,0.357648,-1.588131,3.576279e-07,-1.680781,0.000585,3.104318,NaN,-1.901591
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,-1.633337,-0.003418,0.357648,-3.150200,7.659197e-06,-3.216790,-0.000197,1.822163,NaN,-4.783436
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,-1.464217,-0.000417,0.005252,-4.653107,-1.162291e-05,-3.354657,0.000510,2.667778,NaN,-4.984299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,Y,56960499,726033,A,G,0.0,1.0,NM_001304990.2,36.0,K,...,-0.414166,-0.000091,0.005252,-1.494453,-5.513430e-06,-2.435236,-0.001451,2.540560,NaN,-2.214205
242128,Y,57084754,777848,C,G,0.0,2.0,NM_005638.6,109.0,V,...,-1.074717,-0.000446,0.005252,-1.905708,-1.013279e-06,-1.233571,-0.002534,2.924839,NaN,-4.122275
242129,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,-3.095034,-0.000956,0.357648,-6.131863,-4.357100e-05,-2.936291,-0.000638,2.801300,-3.9775,-3.547984
242130,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,-0.255677,-0.000129,0.020306,-2.886108,5.185604e-06,0.014036,-0.005069,2.687465,NaN,-2.359769


In [8]:
import pandas as pd

def balance_group(group):
    n0 = (group["ClinVar_label"] == 0).sum()
    n1 = (group["ClinVar_label"] == 1).sum()
    
    if n0 == 0 or n1 == 0:
        return pd.DataFrame()  # 只有單一 label 的 group 直接丟棄
    
    minority = min(n0, n1)
    df0 = group[group["ClinVar_label"] == 0].sample(n=minority, random_state=42)
    df1 = group[group["ClinVar_label"] == 1].sample(n=minority, random_state=42)
    return pd.concat([df0, df1])

balanced = (
    ClinVarBenchmark_VCF_PB
    .groupby("ClinVarName_refseq_ids", group_keys=False)
    .apply(balance_group)
    .reset_index(drop=False)  # drop=False 保留 index 變回欄位
    .rename(columns={"index": "ClinVarName_refseq_ids"})
    # 如果上面 rename 沒用，試這個：
    # .reset_index().rename(columns={"level_0": "ClinVarName_refseq_ids"})
)

print(f"Original : {len(ClinVarBenchmark_VCF_PB):,}  (0: {(ClinVarBenchmark_VCF_PB.ClinVar_label==0).sum():,} | 1: {(ClinVarBenchmark_VCF_PB.ClinVar_label==1).sum():,})")
print(f"Balanced : {len(balanced):,}  (0: {(balanced.ClinVar_label==0).sum():,} | 1: {(balanced.ClinVar_label==1).sum():,})")
print(f"Groups retained: {balanced['ClinVarName_refseq_ids'].nunique():,} / {ClinVarBenchmark_VCF_PB['ClinVarName_refseq_ids'].nunique():,}")

balanced

Original : 242,132  (0: 172,617 | 1: 69,515)
Balanced : 96,792  (0: 48,396 | 1: 48,396)
Groups retained: 96,792 / 11,375


,ClinVarName_refseq_ids,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max,vesm_score,gpnstar_v_llr
0,7797,1,75763641,298081,T,C,0.0,2.0,NaN,NaN,...,-0.960893,-0.000391,0.001097,-2.799416,-4.589558e-06,-2.342896,-0.000087,3.122560,NaN,1.418885
1,7734,1,75733592,92263,A,C,0.0,2.0,117.0,T,...,-3.033762,-0.000473,0.005252,-4.101952,-3.802776e-05,-4.355964,0.002521,3.504639,NaN,-3.759413
2,7772,1,75750501,136257,C,T,0.0,2.0,300.0,T,...,-0.336740,-0.000547,0.005252,-1.925934,-1.218915e-05,0.260620,-0.002479,2.685974,NaN,-2.847703
3,7777,1,75760872,1224012,G,C,0.0,1.0,NaN,NaN,...,-1.330051,0.000079,0.011909,-4.111912,-3.874302e-06,-0.327810,0.000345,3.500876,NaN,-1.403370
4,7738,1,75734484,1240439,A,G,0.0,1.0,NaN,NaN,...,0.200759,-0.000341,0.011909,-1.785515,5.662441e-07,-2.752316,0.000096,2.706259,NaN,-1.110384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96787,42960,12,868873,956057,T,G,1.0,1.0,1134.0,Y,...,-1.580957,-0.004778,0.373006,-5.469070,-6.338954e-05,-3.250878,0.004417,5.519279,-18.3190,-10.640390
96788,42959,12,868844,1393975,C,T,1.0,1.0,1125.0,Q,...,-0.069753,-0.002138,0.373006,-6.749489,-6.124377e-05,-1.273228,0.002106,2.505093,-18.3190,-10.513641
96789,42950,12,867907,1069964,C,A,1.0,1.0,812.0,Y,...,-1.256362,-0.004483,0.373006,-4.006607,-5.131960e-05,-3.370415,0.000184,4.303998,-18.3190,-8.107444
96790,147982,22,23767600,473429,G,C,0.0,2.0,NaN,NaN,...,-2.053552,-0.002664,0.011909,-3.403217,-1.403689e-05,-3.107627,0.001551,6.248081,NaN,-1.358093


In [9]:
balanced.to_csv("./clinvar_balanced.csv", index=False)

In [10]:
import pandas as pd

ClinVarBenchmark_VCF_PB = pd.read_csv("./clinvar_updated.csv", low_memory=False)
ClinVarBenchmark_VCF_PB

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max,vesm_score,gpnstar_v_llr
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,-1.801111,-0.001966,0.357648,-1.033955,-1.668930e-06,-1.666055,0.000043,2.827893,NaN,-5.484859
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,-0.810768,-0.001260,0.357648,-1.569786,4.708767e-06,-1.588054,0.000701,3.240415,NaN,-3.155202
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,-0.815928,0.000659,0.357648,-1.588131,3.576279e-07,-1.680781,0.000585,3.104318,NaN,-1.901591
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,-1.633337,-0.003418,0.357648,-3.150200,7.659197e-06,-3.216790,-0.000197,1.822163,NaN,-4.783436
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,-1.464217,-0.000417,0.005252,-4.653107,-1.162291e-05,-3.354657,0.000510,2.667778,NaN,-4.984299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,Y,56960499,726033,A,G,0.0,1.0,NM_001304990.2,36.0,K,...,-0.414166,-0.000091,0.005252,-1.494453,-5.513430e-06,-2.435236,-0.001451,2.540560,NaN,-2.214205
242128,Y,57084754,777848,C,G,0.0,2.0,NM_005638.6,109.0,V,...,-1.074717,-0.000446,0.005252,-1.905708,-1.013279e-06,-1.233571,-0.002534,2.924839,NaN,-4.122275
242129,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,-3.095034,-0.000956,0.357648,-6.131863,-4.357100e-05,-2.936291,-0.000638,2.801300,-3.9775,-3.547984
242130,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,-0.255677,-0.000129,0.020306,-2.886108,5.185604e-06,0.014036,-0.005069,2.687465,NaN,-2.359769


In [11]:
BIN_SIZE = 500_000

ClinVarBenchmark_VCF_PB["bin"] = (
    ClinVarBenchmark_VCF_PB["#CHROM"].astype(str) + "_" +
    (ClinVarBenchmark_VCF_PB["POS"] // BIN_SIZE).astype(str)
)

bin_stats = ClinVarBenchmark_VCF_PB.groupby("bin")["ClinVar_label"].agg(
    total="count",
    n0=lambda x: (x==0).sum(),
    n1=lambda x: (x==1).sum()
)
print(bin_stats["total"].describe())
print(f"\nBins with both labels: {((bin_stats.n0>0) & (bin_stats.n1>0)).sum():,} / {len(bin_stats):,}")

count    4033.000000
mean       60.037689
std       102.486529
min         1.000000
25%         6.000000
50%        26.000000
75%        73.000000
max      1694.000000
Name: total, dtype: float64

Bins with both labels: 2,268 / 4,033


In [12]:
def balance_group(group):
    n0 = (group["ClinVar_label"] == 0).sum()
    n1 = (group["ClinVar_label"] == 1).sum()
    if n0 == 0 or n1 == 0:
        return pd.DataFrame()
    minority = min(n0, n1)
    df0 = group[group["ClinVar_label"] == 0].sample(n=minority, random_state=42)
    df1 = group[group["ClinVar_label"] == 1].sample(n=minority, random_state=42)
    return pd.concat([df0, df1])

balanced_genomic = (
    ClinVarBenchmark_VCF_PB
    .groupby("bin", group_keys=False)
    .apply(balance_group)
)

# bin 是原本就在 df 裡的欄位，apply 後應該還在
# 如果 bin 跑到 index，這行會把它還原
if "bin" not in balanced_genomic.columns:
    balanced_genomic = balanced_genomic.reset_index(level=0, names=["bin"])

balanced_genomic = balanced_genomic.reset_index(drop=True)

print(f"Original : {len(ClinVarBenchmark_VCF_PB):,}  (0: {(ClinVarBenchmark_VCF_PB.ClinVar_label==0).sum():,} | 1: {(ClinVarBenchmark_VCF_PB.ClinVar_label==1).sum():,})")
print(f"Balanced : {len(balanced_genomic):,}  (0: {(balanced_genomic.ClinVar_label==0).sum():,} | 1: {(balanced_genomic.ClinVar_label==1).sum():,})")
print(f"Bins retained: {balanced_genomic['bin'].nunique():,} / {ClinVarBenchmark_VCF_PB['bin'].nunique():,}")

Original : 242,132  (0: 172,617 | 1: 69,515)
Balanced : 111,508  (0: 55,754 | 1: 55,754)
Bins retained: 111,508 / 4,033


In [13]:
balanced_genomic.to_csv("./clinvar_balancedgenome.csv", index=False)